# Arabic Audio Transcriber — Local Notebook

Transcribes Arabic (and mixed Arabic/English) audio using **Faster-Whisper** on your local machine.

**Workflow:**
1. **Batch transcribe** a folder of audio/video files (MP3, MP4, WAV, FLAC, M4A, MKV …). All timestamped **Arabic** transcripts (`[MM:SS-MM:SS]` lines) are saved to `transcript/arabic/<name>.txt`.
2. **Translate** each Arabic transcript into English at `transcript/english/<name>.txt`. This is done locally by the Aya translation stage (see the translation section below); the translation policy lives in `prompts/islamic_translation_runtime.md`. Files stay matched to media by name (`<name>.mp3`, `<name>.mp4`, …).
3. **Batch final videos**: converts every English `<name>.txt` into `<name>.srt`, then renders `Intro.mp4` + looping `Background.mp4` + the audio + burned-in subtitles → `output/<name>_final.mp4`.


---
**Prerequisites (one-time setup):**
- Python ≥ 3.11 with a virtualenv activated
- `pip install faster-whisper yt-dlp torch tqdm`
- PyTorch CUDA build (if using GPU): `pip install torch --index-url https://download.pytorch.org/whl/cu128`
- [ffmpeg](https://ffmpeg.org/download.html) installed and on your PATH

Run **Cell 1** to verify all dependencies are present, then work top to bottom.

# Translation prompt

The translation policy is no longer inlined here. To keep the notebook and the
prompt files from drifting apart, the authoritative prompts live in `prompts/`:

- **`prompts/islamic_translation_canonical.md`** — the full, human-readable specification (governing document).
- **`prompts/islamic_translation_runtime.md`** — the compact runtime prompt the local Aya model is actually sent.

Per-lecture terminology and context can be added at `translation_context/<name>.md`
(matched to a transcript by filename stem). See `translation_context/_TEMPLATE.md`.

In [1]:
# ── Cell 1: Verify dependencies ──────────────────────────────────────────────
import importlib, shutil, sys

missing = [pkg for pkg in ["faster_whisper", "yt_dlp", "torch", "tqdm"] if importlib.util.find_spec(pkg) is None]
if missing:
    print("Missing Python packages:", ", ".join(missing))
    print("Install with:  pip install", " ".join(p.replace('_', '-') for p in missing))
else:
    print("✓ Python packages OK")

if importlib.util.find_spec("hf_xet") is None:
    print("⚠ 'hf_xet' not installed — model downloads from Hugging Face will be slow.")
    print("  Fix:  pip install hf_xet   (then Kernel → Restart)")
else:
    print("✓ hf_xet OK — fast Hugging Face downloads enabled")

if shutil.which("ffmpeg"):
    import subprocess
    v = subprocess.run(["ffmpeg", "-version"], capture_output=True, text=True).stdout.splitlines()[0]
    print(f"✓ ffmpeg: {v}")
else:
    print("✗ ffmpeg not found — install from https://ffmpeg.org/download.html and add it to PATH")

import torch
print(f"Python : {sys.executable}")
print(f"torch  : {torch.__version__}")

if torch.cuda.is_available():
    print(f"✓ GPU  : {torch.cuda.get_device_name(0)}")
elif "+cpu" in torch.__version__:
    print("⚠ CPU-only PyTorch (+cpu) — transcription will use CPU.")
    print("  Fix (in the project .venv, with it activated):")
    print("    pip uninstall torch torchaudio -y")
    print("    pip install torch torchaudio --index-url https://download.pytorch.org/whl/cu126")
    print("  Then: Kernel → Restart, and re-run this cell.")
else:
    print("⚠ torch.cuda.is_available() is False — will use CPU.")
    print("  Run:  python check_cuda.py   from the project folder for details.")

✓ Python packages OK
✓ hf_xet OK — fast Hugging Face downloads enabled
✓ ffmpeg: ffmpeg version 7.1.1-full_build-www.gyan.dev Copyright (c) 2000-2025 the FFmpeg developers
Python : c:\Users\subha\anaconda3\envs\transcribe\python.exe
torch  : 2.5.1+cu121
✓ GPU  : NVIDIA GeForce RTX 3070


## ⚙️ Configuration

Edit the values in the cell below before running anything else.

| Setting | Options | Notes |
|---------|---------|-------|
| `MODEL_SIZE` | `tiny` `base` `small` `medium` `large-v2` | Larger = more accurate, slower |
| `LANGUAGE` | `"ar"`, `"en"`, `"fr"` … or `None` | `None` = auto-detect from first 30 s |
| `START_TIME` | `"8:00"`, `"1:25:00"`, `None` | `None` = from beginning (applies to **every** file in the batch) |
| `END_TIME` | `"25:00"`, `None` | `None` = to end of file (applies to **every** file in the batch) |

In [2]:
# ── Cell 2: Configuration ─────────────────────────────────────────────────────
import re
import torch
from pathlib import Path

MODEL_SIZE = "large-v3-turbo"   # tiny | base | small | medium | large-v2
LANGUAGE   = "ar"          # e.g. "ar", "en", "fr", or None to auto-detect

# Optional time range — set to None to transcribe the whole file
START_TIME = None          # e.g. "8:00" or "1:25:00"
END_TIME   = None          # e.g. "25:00"

# YouTube video download: max height in pixels (720, 1080, …). Used in Cell 3B.
MAX_VIDEO_HEIGHT = 720

# ── Transcript storage (Phase 4: keep the Arabic source separate from English) ─
# Arabic (Faster-Whisper output) and English (translated) transcripts live in
# separate folders so the original Arabic transcript is never overwritten by a
# translation. Filenames share the same stem across every stage:
#   input/<name>.mp3 → transcript/arabic/<name>.txt
#                    → transcript/english/<name>.txt → output/<name>_final.mp4
ARABIC_TRANSCRIPTS_DIR  = Path("transcript/arabic")
ENGLISH_TRANSCRIPTS_DIR = Path("transcript/english")
TRANSLATION_RUNS_DIR    = Path("translation_runs")

for _d in (ARABIC_TRANSCRIPTS_DIR, ENGLISH_TRANSCRIPTS_DIR, TRANSLATION_RUNS_DIR):
    _d.mkdir(parents=True, exist_ok=True)

# Backwards-compatible alias: the batch transcription cell writes Arabic output
# through TRANSCRIPTS_DIR, which now points at transcript/arabic/.
TRANSCRIPTS_DIR = ARABIC_TRANSCRIPTS_DIR

# ── Helpers ───────────────────────────────────────────────────────────────────
DEVICE       = "cuda" if torch.cuda.is_available() else "cpu"
COMPUTE_TYPE = "float16" if DEVICE == "cuda" else "int8"

def _parse_ts(s):
    """Parse 'SS', 'M:SS', 'MM:SS', or 'H:MM:SS' into seconds (float)."""
    if s is None:
        return None
    parts = str(s).strip().split(":")
    nums  = [float(p) for p in parts]
    if len(nums) == 1: return nums[0]
    if len(nums) == 2: return nums[0] * 60 + nums[1]
    return nums[0] * 3600 + nums[1] * 60 + nums[2]

def _fmt_ts(t):
    """Format seconds as MM:SS or H:MM:SS."""
    s = int(round(t))
    h, r = divmod(s, 3600)
    m, sec = divmod(r, 60)
    return f"{h}:{m:02d}:{sec:02d}" if h else f"{m:02d}:{sec:02d}"

def _slugify(text: str, max_len: int = 60) -> str:
    """Filesystem-safe name (keeps letters/digits/Arabic, collapses spaces to _)."""
    text = text.strip().lower()
    text = re.sub(r"[^\w\s-]", "", text, flags=re.UNICODE)
    text = re.sub(r"[\s_-]+", "_", text)
    return text.strip("_")[:max_len] or "output"

def _unique_dir(base: Path) -> Path:
    """Return base if unused, otherwise base_2, base_3, …"""
    if not base.exists():
        return base
    i = 2
    while True:
        candidate = base.parent / f"{base.name}_{i}"
        if not candidate.exists():
            return candidate
        i += 1

start_sec = _parse_ts(START_TIME)
end_sec   = _parse_ts(END_TIME)

print(f"Model   : {MODEL_SIZE}  |  Device: {DEVICE}  |  Compute: {COMPUTE_TYPE}")
print(f"Language: {LANGUAGE or 'auto-detect'}")
if start_sec is not None or end_sec is not None:
    s = _fmt_ts(start_sec or 0)
    e = _fmt_ts(end_sec) if end_sec is not None else "end"
    print(f"Range   : {s} → {e}")
else:
    print("Range   : full file")

Model   : large-v3-turbo  |  Device: cuda  |  Compute: float16
Language: ar
Range   : full file


## 📦 Model cache

Downloads `MODEL_SIZE` from Hugging Face **once** and reuses the local cache on every later run — no re-download, no network call, unless you explicitly ask to check for updates.

In [3]:
# ── Cell 2B: Model cache — download once, reuse offline ──────────────────────
#
# First run of a new MODEL_SIZE: downloads the model (this is the slow part —
# install `hf_xet` per Cell 1 to speed it up).
# Every later run: loads straight from the local Hugging Face cache with
# local_files_only=True, so there is zero network traffic and zero delay.
#
# Set CHECK_FOR_MODEL_UPDATES = True and re-run this cell to check Hugging Face
# for a newer revision of the model (only re-downloads if something changed).

from pathlib import Path
from faster_whisper.utils import download_model

CHECK_FOR_MODEL_UPDATES = False


def _is_complete(model_dir) -> bool:
    """local_files_only=True happily returns a snapshot dir even if the big
    weights file never finished downloading (e.g. a previous run was
    interrupted). Verify model.bin actually resolves before trusting it."""
    p = Path(model_dir) / "model.bin"
    return p.exists() and p.stat().st_size > 0


MODEL_PATH = None
if not CHECK_FOR_MODEL_UPDATES:
    try:
        candidate = download_model(MODEL_SIZE, local_files_only=True)
        if _is_complete(candidate):
            MODEL_PATH = candidate
            print(f"✓ Model '{MODEL_SIZE}' ready (local cache): {MODEL_PATH}")
        else:
            print(f"'{MODEL_SIZE}' is cached but incomplete (missing/empty model.bin) — will re-download.")
    except Exception:
        pass

if MODEL_PATH is None:
    print(f"Downloading '{MODEL_SIZE}' (one-time — this is the slow part; hf_xet speeds it up)…")
    MODEL_PATH = download_model(MODEL_SIZE, local_files_only=False)
    if not _is_complete(MODEL_PATH):
        raise RuntimeError(f"Download finished but model.bin is still missing/empty in {MODEL_PATH}")
    print(f"✓ Downloaded and cached: {MODEL_PATH}")

✓ Model 'large-v3-turbo' ready (local cache): C:\Users\subha\.cache\huggingface\hub\models--mobiuslabsgmbh--faster-whisper-large-v3-turbo\snapshots\0a363e9161cbc7ed1431c9597a8ceaf0c4f78fcf


## 📥 Batch transcription

Set `INPUT_FOLDER` in the cell below to a folder of audio/video files. Every file in the folder is transcribed **in succession**, and all timestamped **Arabic** transcripts are saved to:

```
transcript/arabic/<name>.txt
```

Transcript filenames match the media filenames exactly (`test.mp3` → `transcript/arabic/test.txt`). The English translations produced by the translation stage are written to `transcript/english/<name>.txt`, and the final-video cell at the bottom reads only that English folder — so the original Arabic transcript is never overwritten.

In [4]:
# ── Cell 3: Batch transcribe a folder ─────────────────────────────────────────
#
# Transcribes every audio/video file in INPUT_FOLDER, one after another, and
# saves all timestamped transcripts directly in TRANSCRIPTS_DIR:
#   transcript/<name>.txt        (test.mp3 → transcript/test.txt)
#
# Files that already have a transcript are skipped — set OVERWRITE = True to
# re-transcribe everything.

from faster_whisper import WhisperModel

INPUT_FOLDER = r"input"   # ← folder containing the audio/video files
OVERWRITE    = False      # True = re-transcribe files that already have a transcript

MEDIA_EXTS = {
    ".mp3", ".wav", ".m4a", ".flac", ".aac", ".ogg", ".opus", ".wma",
    ".mp4", ".mkv", ".mov", ".webm", ".avi", ".m4v",
}

in_dir = Path(INPUT_FOLDER)
if not in_dir.is_dir():
    raise FileNotFoundError(f"Folder not found: {in_dir.resolve()}")

media_files = sorted(
    f for f in in_dir.iterdir()
    if f.is_file() and f.suffix.lower() in MEDIA_EXTS
)
if not media_files:
    raise FileNotFoundError(f"No audio/video files found in: {in_dir.resolve()}")

TRANSCRIPTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Folder      : {in_dir.resolve()}")
print(f"Transcripts : {TRANSCRIPTS_DIR.resolve()}")
print(f"Files       : {len(media_files)}")
for f in media_files:
    print(f"  • {f.name}  ({f.stat().st_size / 1e6:.1f} MB)")

# ── Load model once for the whole batch ───────────────────────────────────────
print(f"\nLoading Faster-Whisper '{MODEL_SIZE}' on {DEVICE} ({COMPUTE_TYPE}) …")
model = WhisperModel(MODEL_PATH, device=DEVICE, compute_type=COMPUTE_TYPE)

# Optional time range from Cell 2 (applies to every file in the batch)
use_clip   = start_sec is not None or end_sec is not None
clip_start = start_sec or 0.0
if not use_clip:
    clip_timestamps = "0"
    vad_filter = True
elif end_sec is None:
    clip_timestamps = str(clip_start)
    vad_filter = False
else:
    clip_timestamps = f"{clip_start},{float(end_sec)}"
    vad_filter = False

# ── Transcribe each file in succession ────────────────────────────────────────
done, skipped, failed = [], [], []

for idx, media in enumerate(media_files, 1):
    out_txt = TRANSCRIPTS_DIR / f"{media.stem}.txt"
    print(f"\n{'─' * 70}\n[{idx}/{len(media_files)}] {media.name}")

    if out_txt.exists() and not OVERWRITE:
        print(f"  ↷ Skipped — transcript already exists: {out_txt}")
        skipped.append(media.name)
        continue

    try:
        segments, info = model.transcribe(
            str(media),
            language=LANGUAGE,
            beam_size=5,
            temperature=0.0,
            condition_on_previous_text=False,
            vad_filter=vad_filter,
            clip_timestamps=clip_timestamps,

            # Helpful for long-form transcription
            word_timestamps=False,
            compression_ratio_threshold=2.4,
            log_prob_threshold=-1.0,
            no_speech_threshold=0.6,
        )

        duration = getattr(info, "duration", None)
        if duration:
            print(f"  Audio length: {_fmt_ts(duration)}")

        timed_lines = []
        for seg in segments:
            text = seg.text.strip()
            if not text:
                continue
            line = f"[{_fmt_ts(float(seg.start or 0))}-{_fmt_ts(float(seg.end or 0))}] {text}"
            timed_lines.append(line)
            print(f"  {line}")

        out_txt.write_text("\n".join(timed_lines), encoding="utf-8")
        print(f"  ✓ Saved {len(timed_lines)} segments → {out_txt}")
        done.append(media.name)

    except Exception as e:
        print(f"  ✗ FAILED: {e}")
        failed.append(media.name)

# ── Summary ───────────────────────────────────────────────────────────────────
print(f"\n{'═' * 70}")
print(f"✓ Transcribed: {len(done)}   ↷ Skipped: {len(skipped)}   ✗ Failed: {len(failed)}")
for name in failed:
    print(f"  ✗ {name}")
print(f"Transcripts saved in: {TRANSCRIPTS_DIR.resolve()}")

# ── Release the Faster-Whisper model before translation (Phase 3 GPU rule) ────
# The RTX 3070 has only 8 GB VRAM, so only one large GPU workload should be
# resident at a time. Free Whisper here before starting the Aya vLLM server.
# torch.cuda.empty_cache() does not manage every CTranslate2 allocation, but
# deleting the WhisperModel object and collecting garbage releases its weights.
import gc

try:
    del model
except NameError:
    pass
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print("Released the Faster-Whisper model before translation.")

Folder      : C:\Users\subha\Desktop\transcribe\youtube-translator\input
Transcripts : C:\Users\subha\Desktop\transcribe\youtube-translator\transcript
Files       : 3
  • 1min.mp3  (1.3 MB)
  • 4min.mp3  (4.0 MB)
  • hard.mp3  (3.2 MB)

Loading Faster-Whisper 'large-v3-turbo' on cuda (float16) …

──────────────────────────────────────────────────────────────────────
[1/3] 1min.mp3
  Audio length: 00:53
  [00:00-00:15] لحكم أقرأ القرآن عند جنابة عندي إشكال يعني عند النوم وأنا في جنابة كيف أقرأ القرآن كيف أقرأ آخر آية أذكار النوم يعني
  [00:15-00:28] نعم أما الجنوب فباتفق أربعة وهو قول عمر وعلي ثبت عن علي عنده بشيبة وعمر كما في الخلافيات البيهقي قال علي أما الجنوب فلا ولو آية
  [00:28-00:31] وهذا بخلاف ابن عباس
  [00:31-00:32] فقد علق عن البخاري
  [00:32-00:33] عباس الجنوب يقرأ القرآن
  [00:33-00:36] وهو قول جماعة كالبخاري وغيره
  [00:36-00:37] والصواب أن الجنوب لا يقرأ القرآن
  [00:37-00:40] أما أذكار النوم فهي قسمان
  [00:40-00:42] القسم الأول أذكار
  [00:42-00:43] والقسم الثاني قرآن
 

## 🌐 Local translation (Arabic → English)

This stage translates each `transcript/arabic/<name>.txt` into
`transcript/english/<name>.txt` using **Aya Expanse 8B**, served locally through
a **vLLM** OpenAI-compatible server in WSL2. The original Arabic transcript is
never modified.

**Before running the translation cells:**

1. Make sure the Faster-Whisper model has been released (the transcription cell
   does this automatically at the end).
2. Start the vLLM server in WSL2 (see `docs/wsl_vllm_setup.md` and
   `docs/gpu_operating_rules.md`). Only one large GPU workload should be resident
   at a time on the 8 GB RTX 3070.

The configuration cell below defines the backend, model, prompt paths, run
flags, context budget, and deterministic generation settings. Cells 5-7 below implement the timestamp parser, the token-aware chunker, and the
vLLM translation backend; Cell 7B health-checks the server. Cells 7C-7E add strict
validation, retry/recovery, and chunk checkpointing with resume (Phases 11-13);
Cell 7F is the single-chunk smoke test (Phase 15), which must pass in the current
kernel session before Cell 7G runs the folder-level batch translation (Phase 14).

In [ ]:
# ── Cell 4: Translation configuration (Phase 6) ───────────────────────────────
from pathlib import Path

# Backend options:
#   "aya_vllm"  — Aya Expanse 8B served locally through vLLM (WSL2)
TRANSLATION_BACKEND = "aya_vllm"

AYA_MODEL_ID = "CohereLabs/aya-expanse-8b"

# Local vLLM OpenAI-compatible server (started in WSL2 — see docs/wsl_vllm_setup.md)
VLLM_BASE_URL = "http://localhost:8000/v1"
VLLM_API_KEY  = "local"

# Transcript + run directories (kept in sync with the transcription config cell)
ARABIC_TRANSCRIPTS_DIR  = Path("transcript/arabic")
ENGLISH_TRANSCRIPTS_DIR = Path("transcript/english")
TRANSLATION_RUNS_DIR    = Path("translation_runs")

# Prompt + per-lecture context files
CANONICAL_PROMPT_PATH = Path("prompts/islamic_translation_canonical.md")
RUNTIME_PROMPT_PATH   = Path("prompts/islamic_translation_runtime.md")
CONTEXT_DIR           = Path("translation_context")

for _d in (ARABIC_TRANSCRIPTS_DIR, ENGLISH_TRANSCRIPTS_DIR, TRANSLATION_RUNS_DIR, CONTEXT_DIR):
    _d.mkdir(parents=True, exist_ok=True)

# ── Run behaviour ─────────────────────────────────────────────────────────────
OVERWRITE_TRANSLATIONS = False   # re-translate files that already have English output
RESUME_TRANSLATIONS    = True    # reuse validated chunk checkpoints when inputs match
MAX_RETRIES            = 2       # retry attempts per chunk before marking it failed
CONTINUE_ON_ERROR      = False   # keep going to the next file after an unrecoverable failure

# ── Context budget ────────────────────────────────────────────────────────────
# MUST match the running server's --max-model-len exactly (Cell 7B verifies
# this live and raises a clear error on mismatch). Conservative for an 8 GB
# RTX 3070 running Aya at 4-bit; chunking is token- and timestamp-block aware
# (never a fixed 20-minute chunk). Currently matches the OOM-fallback launch
# command in docs/wsl_vllm_setup.md (--max-model-len 2560) — raise this (and
# the server flag together) only if you restart the server with more room.
AYA_CONTEXT_LIMIT = 2560

# Aya prompt packaging: "system" or "single_user".
# Decided by the live smoke test (Phase 15): single_user made Aya transliterate
# and emit markdown; system mode translated cleanly. See todo.md Phase 5.
AYA_PROMPT_MODE = "system"

# ── Generation policy ─────────────────────────────────────────────────────────
# Deterministic generation for reproducible, testable translations.
TEMPERATURE = 0.0
TOP_P       = 1.0


def load_optional_context(stem: str) -> str:
    """Return the per-lecture context for ``<stem>.md`` if present, else "".

    Files whose name begins with "_" (e.g. _TEMPLATE.md) are ignored so the
    template does not accidentally match a transcript stem. This context is used
    only to resolve ambiguity and keep terminology consistent; it is never
    inserted into the translation.
    """
    if stem.startswith("_"):
        return ""
    path = CONTEXT_DIR / f"{stem}.md"
    if path.is_file():
        return path.read_text(encoding="utf-8").strip()
    return ""


# ── Load the runtime prompt + report its token footprint ──────────────────────
if not RUNTIME_PROMPT_PATH.is_file():
    raise FileNotFoundError(
        f"Runtime prompt not found: {RUNTIME_PROMPT_PATH.resolve()}\n"
        "Create prompts/islamic_translation_runtime.md (Phase 5)."
    )

runtime_prompt = RUNTIME_PROMPT_PATH.read_text(encoding="utf-8").strip()

# Rough heuristics only — the exact Aya token count is measured in the
# token-aware chunking cell (Phase 8) using the Aya tokenizer.
_char_count = len(runtime_prompt)
_word_count = len(runtime_prompt.split())
_approx_by_chars = round(_char_count / 4)
_approx_by_words = round(_word_count * 1.33)

print(f"Backend        : {TRANSLATION_BACKEND}")
print(f"Model          : {AYA_MODEL_ID}")
print(f"vLLM URL       : {VLLM_BASE_URL}")
print(f"Context limit  : {AYA_CONTEXT_LIMIT} tokens")
print(f"Prompt mode    : {AYA_PROMPT_MODE}")
print(f"Generation     : temperature={TEMPERATURE}, top_p={TOP_P}")
print(f"Runtime prompt : {RUNTIME_PROMPT_PATH}")
print(
    f"  size         : {_char_count} chars, {_word_count} words "
    f"(≈{_approx_by_words}-{_approx_by_chars} tokens by heuristic; target 800-1,500)"
)
print("Note: exact Aya token count is verified in the chunking cell (Phase 8).")

### 🧩 Timestamp parser (Phase 7)

Parses `[START-END] text` transcript lines into `TimestampBlock` objects and rebuilds
them with `blocks_to_text`. Timestamps are preserved **exactly as written** — never
normalized — because validation (Phase 11) requires the translated timestamp sequence
to match the source character-for-character. Malformed lines raise immediately instead
of being silently dropped. The cell runs its own self-tests every time it executes.

In [ ]:
# ── Cell 5: Timestamp parser (Phase 7) ────────────────────────────────────────
#
# Foundation of the translation stage. Parses "[START-END] text" transcript
# lines into TimestampBlock objects and rebuilds transcripts from blocks.
# The timestamp string is preserved EXACTLY as written — never normalized —
# because validation (Phase 11) requires the translated timestamp sequence to
# match the source character-for-character. Malformed lines fail loudly rather
# than being silently dropped.
import re
from dataclasses import dataclass

TIMESTAMP_LINE_RE = re.compile(
    r"^(?P<timestamp>\[[^\[\]\r\n]+-[^\[\]\r\n]+\])\s*(?P<text>.*)$"
)


@dataclass(frozen=True)
class TimestampBlock:
    timestamp: str            # exact source token, e.g. "[00:07-00:12]"
    text: str
    source_line_number: int


def parse_timestamped_transcript(text: str) -> list[TimestampBlock]:
    """Parse a timestamped transcript into blocks.

    Every non-blank line must start with a "[START-END]" timestamp; anything
    else raises ValueError so a broken transcript is caught before it reaches
    the model or the final English output.
    """
    blocks: list[TimestampBlock] = []

    for line_number, raw_line in enumerate(text.splitlines(), start=1):
        line = raw_line.strip()
        if not line:
            continue

        match = TIMESTAMP_LINE_RE.match(line)
        if not match:
            raise ValueError(
                f"Line {line_number} is not a valid timestamp block: {raw_line!r}"
            )

        blocks.append(
            TimestampBlock(
                timestamp=match.group("timestamp"),
                text=match.group("text").strip(),
                source_line_number=line_number,
            )
        )

    if not blocks:
        raise ValueError("Transcript contains no timestamp blocks.")

    return blocks


def blocks_to_text(blocks: list[TimestampBlock]) -> str:
    """Rebuild transcript text from blocks (one "[START-END] text" line each)."""
    return "\n".join(
        f"{block.timestamp} {block.text}".rstrip()
        for block in blocks
    )


# ── Parser self-tests (the Phase 7 required cases) ────────────────────────────
def _run_parser_tests() -> None:
    # Standard MM:SS timestamp
    blocks = parse_timestamped_transcript("[00:00-00:05] بسم الله الرحمن الرحيم")
    assert blocks[0].timestamp == "[00:00-00:05]"
    assert blocks[0].text == "بسم الله الرحمن الرحيم"
    assert blocks[0].source_line_number == 1

    # Hour timestamp (H:MM:SS, as produced by _fmt_ts in Cell 2)
    blocks = parse_timestamped_transcript("[1:02:03-1:02:10] الحمد لله")
    assert blocks[0].timestamp == "[1:02:03-1:02:10]"

    # Arabic punctuation in the text
    blocks = parse_timestamped_transcript("[00:05-00:11] ما حكم هذا؟ قال: لا بأس به،")
    assert blocks[0].text == "ما حكم هذا؟ قال: لا بأس به،"

    # Empty source text after a timestamp is allowed (kept as an empty block)
    blocks = parse_timestamped_transcript("[00:11-00:15]")
    assert blocks[0].text == ""

    # Blank lines between blocks are skipped; line numbers track the source
    blocks = parse_timestamped_transcript("[00:00-00:05] أولا\n\n\n[00:05-00:11] ثانيا")
    assert len(blocks) == 2
    assert blocks[1].source_line_number == 4

    # A malformed line (no timestamp) is rejected loudly
    try:
        parse_timestamped_transcript("[00:00-00:05] أولا\nهذا سطر بلا توقيت")
    except ValueError as e:
        assert "Line 2" in str(e)
    else:
        raise AssertionError("Malformed line was not rejected")

    # An empty transcript is rejected
    try:
        parse_timestamped_transcript("\n  \n")
    except ValueError:
        pass
    else:
        raise AssertionError("Empty transcript was not rejected")

    # Duplicate timestamps are preserved, not deduplicated
    blocks = parse_timestamped_transcript("[00:00-00:05] الأول\n[00:00-00:05] الثاني")
    assert [b.timestamp for b in blocks] == ["[00:00-00:05]", "[00:00-00:05]"]

    # Leading zeros are preserved exactly — no normalization
    blocks = parse_timestamped_transcript("[00:07-00:09] نعم\n[0:07-0:09] نعم")
    assert blocks[0].timestamp == "[00:07-00:09]"
    assert blocks[1].timestamp == "[0:07-0:09]"

    # Round trip: parse → rebuild is lossless (modulo blank lines/whitespace)
    source = "[00:00-00:05] بسم الله\n[00:05-00:11] الحمد لله\n[00:11-00:15]"
    assert blocks_to_text(parse_timestamped_transcript(source)) == source

    print("Timestamp parser: all self-tests passed.")


_run_parser_tests()

### 📏 Token-aware chunking (Phase 8)

Loads the **Aya tokenizer only** (the model itself runs inside the vLLM server) and
plans chunks so that the runtime prompt + lecture context + continuity context +
Arabic source + the reserved English output always fit within `AYA_CONTEXT_LIMIT`,
with a safety margin. Chunks are packed greedily from whole timestamp blocks — a
block is never split. If the budget cannot fit any source text, the cell fails loudly
with a full token breakdown; nothing is truncated silently.

> **Gated model:** downloading the tokenizer requires the Aya Expanse 8B license to
> be accepted on Hugging Face and `hf auth login` on Windows.

In [ ]:
# ── Cell 6: Token-aware chunking (Phase 8) ────────────────────────────────────
#
# Plans translation chunks against the model context, which must hold:
#   runtime prompt + lecture context + continuity context + Arabic source
#   + the generated English output + a safety margin.
# Chunks are built greedily from WHOLE timestamp blocks — a block is never
# split. If the fixed prompt parts leave no room for source text, this fails
# loudly with a full token breakdown; nothing is ever truncated silently.
from dataclasses import dataclass

from transformers import AutoTokenizer

SAFETY_MARGIN_TOKENS      = 160   # tokenizer/template estimation error + the
                                  # Phase 12 retry-reminder's ~41 extra tokens
CONTINUITY_RESERVE_TOKENS = 400   # planning reserve for the continuity context
CONTINUITY_MAX_BLOCKS     = 3     # tail blocks carried between chunks
MAX_OUTPUT_TOKENS         = 1400  # hard cap on the per-chunk English reserve

# ── Load ONLY the tokenizer for planning (the model itself runs in vLLM) ──────
try:
    translation_tokenizer = AutoTokenizer.from_pretrained(AYA_MODEL_ID)
except Exception as e:
    raise RuntimeError(
        "Could not load the Aya tokenizer for chunk planning.\n"
        "Aya is gated: accept the model terms on "
        f"https://huggingface.co/{AYA_MODEL_ID} and run `hf auth login` on "
        "Windows with a token that can download gated models."
    ) from e


def count_tokens(text: str) -> int:
    return len(translation_tokenizer(text, add_special_tokens=False)["input_ids"])


# ── Prompt packaging (Phase 5.4) — shared by the chunk planner and backend ────
def build_task_body(lecture_context: str, previous_context: str, source_text: str) -> str:
    """The per-chunk task sections, identical in every prompt mode."""
    return (
        "LECTURE-SPECIFIC CONTEXT\n"
        f"{lecture_context or '(none)'}\n\n"
        "CONTINUITY CONTEXT — DO NOT OUTPUT\n"
        f"{previous_context or '(none)'}\n\n"
        "CURRENT SOURCE — TRANSLATE ONLY THESE BLOCKS\n"
        f"{source_text}"
    )


def build_chat_messages(
    runtime_prompt: str,
    lecture_context: str,
    previous_context: str,
    source_text: str,
    mode: str | None = None,
) -> list[dict]:
    """Package the prompt per AYA_PROMPT_MODE ("single_user" default, "system")."""
    mode = mode or AYA_PROMPT_MODE
    body = build_task_body(lecture_context, previous_context, source_text)
    if mode == "single_user":
        return [{"role": "user", "content": runtime_prompt + "\n\n" + body}]
    if mode == "system":
        return [
            {"role": "system", "content": runtime_prompt},
            {"role": "user", "content": body},
        ]
    raise ValueError(f"Unknown AYA_PROMPT_MODE: {mode!r}")


def measure_prompt_tokens(
    runtime_prompt: str,
    lecture_context: str,
    previous_context: str,
    source_text: str,
    mode: str | None = None,
) -> int:
    """Token count of the fully rendered chat prompt (template overhead included)."""
    rendered = translation_tokenizer.apply_chat_template(
        build_chat_messages(runtime_prompt, lecture_context, previous_context, source_text, mode),
        tokenize=False,
        add_generation_prompt=True,
    )
    return count_tokens(rendered)


def estimate_output_tokens(source_tokens: int) -> int:
    """Per-chunk English output reserve — English runs longer than the Arabic."""
    return min(MAX_OUTPUT_TOKENS, max(256, int(source_tokens * 1.8) + 128))


# ── Chunk budget (Phase 8.4) ──────────────────────────────────────────────────
class PromptTooLargeError(RuntimeError):
    pass


def _format_budget(budget: dict) -> str:
    return (
        f"  Context limit          : {budget['context_limit']}\n"
        f"  Policy tokens          : {budget['policy_tokens']}\n"
        f"  Lecture-context tokens : {budget['lecture_context_tokens']}\n"
        f"  Rendered overhead      : {budget['rendered_overhead_tokens']} (policy + context + chat template)\n"
        f"  Continuity reserve     : {budget['continuity_reserve_tokens']}\n"
        f"  Reserved output tokens : {budget['max_output_reserve_tokens']}\n"
        f"  Safety margin          : {budget['safety_margin_tokens']}\n"
        f"  Remaining source budget: {budget['max_source_tokens']}"
    )


def plan_source_budget(
    runtime_prompt: str,
    lecture_context: str = "",
    context_limit: int | None = None,
) -> dict:
    """Work out how many Arabic source tokens fit in one chunk.

    Measures the rendered prompt with empty source/continuity, reserves the
    continuity budget, the estimated English output, and a safety margin, then
    finds the largest source size that still fits (output reserve grows with
    the source, so this is a binary search rather than one subtraction).
    """
    context_limit = context_limit or AYA_CONTEXT_LIMIT
    rendered_overhead = measure_prompt_tokens(runtime_prompt, lecture_context, "", "")

    def _fits(source_tokens: int) -> bool:
        return (
            rendered_overhead
            + CONTINUITY_RESERVE_TOKENS
            + source_tokens
            + estimate_output_tokens(source_tokens)
            + SAFETY_MARGIN_TOKENS
        ) <= context_limit

    lo, hi = 0, context_limit
    while lo < hi:
        mid = (lo + hi + 1) // 2
        if _fits(mid):
            lo = mid
        else:
            hi = mid - 1
    max_source_tokens = lo

    budget = {
        "context_limit": context_limit,
        "policy_tokens": count_tokens(runtime_prompt),
        "lecture_context_tokens": count_tokens(lecture_context) if lecture_context else 0,
        "rendered_overhead_tokens": rendered_overhead,
        "continuity_reserve_tokens": CONTINUITY_RESERVE_TOKENS,
        "safety_margin_tokens": SAFETY_MARGIN_TOKENS,
        "max_source_tokens": max_source_tokens,
        "max_output_reserve_tokens": estimate_output_tokens(max_source_tokens),
    }
    if max_source_tokens <= 0:
        raise PromptTooLargeError(
            "The fixed prompt parts leave no room for Arabic source text.\n"
            + _format_budget(budget)
            + "\nShorten the runtime prompt / lecture context, or raise "
            "AYA_CONTEXT_LIMIT together with the server's --max-model-len."
        )
    return budget


# ── Greedy block chunking (Phase 8.5) ─────────────────────────────────────────
@dataclass
class TranslationChunk:
    index: int                      # 1-based position within the transcript
    blocks: list[TimestampBlock]
    source_text: str
    source_tokens: int
    max_new_tokens: int             # English output reserve for this chunk


def build_chunks(
    blocks: list[TimestampBlock],
    runtime_prompt: str,
    lecture_context: str = "",
    context_limit: int | None = None,
) -> list[TranslationChunk]:
    """Greedily pack whole timestamp blocks into token-budgeted chunks."""
    budget = plan_source_budget(runtime_prompt, lecture_context, context_limit)
    max_source = budget["max_source_tokens"]

    chunks: list[TranslationChunk] = []
    current: list[TimestampBlock] = []
    current_tokens = 0

    def _flush() -> None:
        nonlocal current, current_tokens
        if not current:
            return
        source_text = blocks_to_text(current)
        source_tokens = count_tokens(source_text)
        chunks.append(
            TranslationChunk(
                index=len(chunks) + 1,
                blocks=current,
                source_text=source_text,
                source_tokens=source_tokens,
                max_new_tokens=estimate_output_tokens(source_tokens),
            )
        )
        current, current_tokens = [], 0

    for block in blocks:
        line_tokens = count_tokens(f"{block.timestamp} {block.text}".rstrip()) + 1  # +1 newline
        if line_tokens > max_source:
            # Never silently split one block into invented timestamps (8.5).
            raise PromptTooLargeError(
                f"Source line {block.source_line_number} alone is {line_tokens} tokens, "
                f"more than the per-chunk source budget of {max_source}. Refusing to "
                "split a timestamp block. Raise AYA_CONTEXT_LIMIT (and the server's "
                "--max-model-len), or translate this block alone with a reduced prompt."
            )
        if current and current_tokens + line_tokens > max_source:
            _flush()
        current.append(block)
        current_tokens += line_tokens
    _flush()
    return chunks


# ── Continuity context between chunks (Phase 8.6) ─────────────────────────────
def build_continuity_context(
    source_blocks: list[TimestampBlock],
    translated_text: str,
    max_blocks: int = CONTINUITY_MAX_BLOCKS,
    max_tokens: int = CONTINUITY_RESERVE_TOKENS,
) -> str:
    """Small cross-chunk context: the last few source + translated lines.

    Timestamps are stripped so the model cannot mistake context for source
    blocks to translate. Shrinks (and finally returns "") rather than exceed
    max_tokens.
    """
    english_lines = []
    for line in translated_text.splitlines():
        line = line.strip()
        if not line:
            continue
        match = TIMESTAMP_LINE_RE.match(line)
        english_lines.append((match.group("text") if match else line).strip())

    for n in range(max_blocks, 0, -1):
        arabic_tail = [b.text for b in source_blocks[-n:] if b.text]
        english_tail = english_lines[-n:]
        context = (
            "Previous Arabic context:\n" + "\n".join(arabic_tail) + "\n\n"
            "Previous approved English:\n" + "\n".join(english_tail) + "\n\n"
            "This material is context only. Do not output or repeat it.\n"
            "Translate only the timestamped source blocks in CURRENT SOURCE."
        )
        if count_tokens(context) <= max_tokens:
            return context
    return ""


# ── Diagnostics: exact prompt footprint + chunk budget ────────────────────────
print(f"Aya tokenizer loaded : {AYA_MODEL_ID}")
print(f"Runtime prompt       : {count_tokens(runtime_prompt)} tokens exact (target 800-1,500)")

_overhead_single = measure_prompt_tokens(runtime_prompt, "", "", "", mode="single_user")
print(f"Rendered overhead    : {_overhead_single} tokens (single_user mode, empty source)")
try:
    _overhead_system = measure_prompt_tokens(runtime_prompt, "", "", "", mode="system")
    print(f"                       {_overhead_system} tokens (system mode, empty source)")
except Exception as _e:
    print(f"                       system mode failed to render: {_e}")

_budget = plan_source_budget(runtime_prompt)
print(f"Chunk budget         : ≤{_budget['max_source_tokens']} Arabic source tokens per chunk")
print(
    f"                       (context {_budget['context_limit']}"
    f" − overhead {_budget['rendered_overhead_tokens']}"
    f" − continuity {_budget['continuity_reserve_tokens']}"
    f" − output ≤{_budget['max_output_reserve_tokens']}"
    f" − margin {_budget['safety_margin_tokens']})"
)

### 🔌 Translation backend (Phases 9–10)

Defines the backend-agnostic interface (`TranslationResult`, `TranslationBackend`)
and the concrete **`AyaVLLMBackend`**, which sends one deterministic request at a
time (temperature 0) to the local vLLM OpenAI-compatible server. The later pipeline
cells only ever call `translate_chunk()`, so a different local model can be swapped
in without touching the pipeline.

**Start the server first** (in WSL2 — full setup and the out-of-memory fallback
command live in `docs/wsl_vllm_setup.md`):

```bash
source ~/local-translation-vllm/.venv/bin/activate

vllm serve CohereLabs/aya-expanse-8b \
  --host 0.0.0.0 --port 8000 \
  --dtype float16 --quantization bitsandbytes \
  --max-model-len 3072 --max-num-seqs 1 \
  --gpu-memory-utilization 0.90 --enable-prefix-caching
```

Cell 7B below verifies the server is reachable and actually serving the model —
run it after starting the server, and before any translation cell.

In [ ]:
# ── Cell 7: Translation backend — shared interface (Phase 9) + Aya/vLLM (Phase 10)
#
# The pipeline (validation, retries, checkpointing, batch translation) only
# ever calls TranslationBackend.translate_chunk(), so another local or remote
# model can be added later without touching the pipeline.
from dataclasses import dataclass, field
from typing import Protocol

from openai import APIConnectionError, OpenAI


@dataclass
class TranslationResult:
    text: str
    model_id: str
    backend: str
    prompt_tokens: int | None = None
    completion_tokens: int | None = None
    raw_response: object | None = None
    warnings: list[str] = field(default_factory=list)


class TranslationBackend(Protocol):
    model_id: str
    backend_name: str

    def translate_chunk(
        self,
        source_text: str,
        runtime_prompt: str,
        lecture_context: str,
        previous_context: str,
        max_new_tokens: int,
    ) -> TranslationResult:
        ...


def _server_unreachable_message(base_url: str) -> str:
    return (
        f"Cannot reach the local vLLM server at {base_url}.\n"
        "Start the server in WSL2 (see docs/wsl_vllm_setup.md) and verify that "
        "the model loaded successfully."
    )


class AyaVLLMBackend:
    """Aya Expanse 8B through the local vLLM OpenAI-compatible server."""

    backend_name = "aya_vllm"

    def __init__(
        self,
        base_url: str = VLLM_BASE_URL,
        api_key: str = VLLM_API_KEY,
        model_id: str = AYA_MODEL_ID,
    ):
        self.model_id = model_id
        self.base_url = base_url
        # Deterministic generation, one request at a time (Phase 10.4). The
        # HTTP client never retries — retries are a pipeline decision (Phase 12).
        self.client = OpenAI(
            base_url=base_url,
            api_key=api_key,
            timeout=600.0,
            max_retries=0,
        )

    def translate_chunk(
        self,
        source_text: str,
        runtime_prompt: str,
        lecture_context: str = "",
        previous_context: str = "",
        max_new_tokens: int = 1024,
    ) -> TranslationResult:
        # Same packaging the chunk planner measured (Cell 6), so the token
        # budget holds for the request actually sent.
        messages = build_chat_messages(
            runtime_prompt, lecture_context, previous_context, source_text
        )

        try:
            response = self.client.chat.completions.create(
                model=self.model_id,
                messages=messages,
                temperature=TEMPERATURE,
                top_p=TOP_P,
                max_tokens=max_new_tokens,
            )
        except APIConnectionError as e:
            raise RuntimeError(_server_unreachable_message(self.base_url)) from e

        choice = response.choices[0]
        warnings = []
        if choice.finish_reason == "length":
            warnings.append(
                f"Output hit the max_tokens limit ({max_new_tokens}) — likely truncated."
            )

        return TranslationResult(
            text=(choice.message.content or "").strip(),
            model_id=self.model_id,
            backend=self.backend_name,
            prompt_tokens=getattr(response.usage, "prompt_tokens", None),
            completion_tokens=getattr(response.usage, "completion_tokens", None),
            raw_response=response,
            warnings=warnings,
        )


def check_vllm_server(backend: AyaVLLMBackend) -> None:
    """Health check from Windows (Phase 10.2). Raises with a clear message."""
    try:
        models = list(backend.client.models.list())
    except Exception as e:
        raise RuntimeError(_server_unreachable_message(backend.base_url)) from e

    served = [m.id for m in models]
    print(f"vLLM server OK at {backend.base_url} — serving: {', '.join(served)}")
    if backend.model_id not in served:
        raise RuntimeError(
            f"The vLLM server is not serving {backend.model_id!r} (found: {served}).\n"
            "Restart it with the command in docs/wsl_vllm_setup.md."
        )

    # Cross-check AYA_CONTEXT_LIMIT (Cell 4) against the server's actual
    # --max-model-len (vLLM exposes this as `max_model_len` on the model
    # entry). A mismatch means every chunk Cell 6 builds assumes more room
    # than the server actually has — better to fail loudly here than as a
    # BadRequestError mid-batch, several chunks into a lecture.
    served_model = next(m for m in models if m.id == backend.model_id)
    server_max_len = getattr(served_model, "max_model_len", None)
    if server_max_len is not None:
        print(f"Server max_model_len : {server_max_len} tokens")
        if AYA_CONTEXT_LIMIT > server_max_len:
            raise RuntimeError(
                f"AYA_CONTEXT_LIMIT (Cell 4) is {AYA_CONTEXT_LIMIT}, but the running "
                f"server's actual --max-model-len is only {server_max_len}.\n"
                f"Set AYA_CONTEXT_LIMIT = {server_max_len} (or less) in Cell 4 to "
                "match the server you actually started, then re-run Cells 4 onward."
            )


print("Translation backend defined: AyaVLLMBackend "
      f"(model={AYA_MODEL_ID}, url={VLLM_BASE_URL}, mode={AYA_PROMPT_MODE}, "
      f"temperature={TEMPERATURE}, top_p={TOP_P})")

In [ ]:
# ── Cell 7B: vLLM server health check (Phase 10.2) ────────────────────────────
# Run this AFTER starting the vLLM server in WSL2 (docs/wsl_vllm_setup.md).
# It must pass before the smoke-test and batch-translation cells are run.
backend = AyaVLLMBackend(
    base_url=VLLM_BASE_URL,
    api_key=VLLM_API_KEY,
    model_id=AYA_MODEL_ID,
)
check_vllm_server(backend)

### ✅ Strict validation (Phase 11)

Every generated chunk — and the final joined transcript — must pass `validate_translation`
before it is written anywhere. **Errors** (block the output): unparseable output, timestamp
sequence not exactly equal to the source, changed block count, empty translations of
non-empty source blocks, text invented for empty source blocks. **Warnings** (kept for
review in the run artifacts): leftover Arabic script, preambles / markdown / code fences /
notes, refusal language, blocks echoed untranslated, length anomalies, possible truncation.
The cell runs its own self-tests on every execution — no server needed.

In [ ]:
# ── Cell 7C: Strict validation (Phase 11) ─────────────────────────────────────
#
# Every generated chunk — and the final joined transcript — must pass this
# validator before it is written anywhere. Errors block the output entirely;
# warnings flag suspicious-but-possibly-valid output (e.g. a deliberately
# preserved Arabic quotation) and are kept in the run artifacts for review.
import re
from dataclasses import dataclass, field


@dataclass
class ValidationResult:
    valid: bool
    errors: list[str] = field(default_factory=list)
    warnings: list[str] = field(default_factory=list)


# Length-anomaly thresholds (chars of block text, timestamps excluded).
# Arabic → English usually expands ~1.1-1.6×; anything outside is suspect.
LEN_RATIO_SHORT     = 0.7
LEN_RATIO_LONG      = 2.5
LEN_CHECK_MIN_CHARS = 200   # skip the ratio checks on tiny samples

_PREAMBLE_RE = re.compile(
    r"(?i)^\s*(translation\b|here (?:is|are)\b|here's\b|sure\b|certainly\b|of course\b)"
)
_REFUSAL_RE = re.compile(
    r"(?i)\b(i cannot|i can't|i am unable|i'm unable|i'm sorry|as an ai"
    r"|cannot assist|cannot help with)\b"
)
_NOTE_LINE_RE   = re.compile(r"(?i)^\s*\(?\s*(note|n\.b\.)\s*[:\-\)]")
_ARABIC_CHAR_RE = re.compile(r"[\u0600-\u06FF]")
_TRUNCATED_TAIL_CHARS = (",", "،", ":", "-", "—")


def extract_timestamps(text: str) -> list[str]:
    """Timestamps of every line that looks like a timestamp block (lenient —
    used to diagnose output that does not parse as a clean transcript)."""
    timestamps = []
    for line in text.splitlines():
        line = line.strip()
        if not line:
            continue
        match = TIMESTAMP_LINE_RE.match(line)
        if match:
            timestamps.append(match.group("timestamp"))
    return timestamps


def validate_translation(source_text: str, translated_text: str) -> ValidationResult:
    """Strict contract check of translated output against its source.

    Errors (block the output): unparseable output, timestamp sequence not
    exactly equal to the source, changed block count, empty translations of
    non-empty source blocks, text invented for empty source blocks.
    Warnings (kept for review): leftover Arabic script, preambles / markdown /
    code fences / notes, refusal language, blocks echoed untranslated,
    length anomalies, possible truncation.
    """
    errors: list[str] = []
    warnings: list[str] = []

    source_blocks = parse_timestamped_transcript(source_text)
    source_timestamps = [b.timestamp for b in source_blocks]

    try:
        translated_blocks = parse_timestamped_transcript(translated_text)
    except ValueError as e:
        translated_blocks = None
        errors.append(f"Output is not a clean timestamped transcript: {e}")

    translated_timestamps = (
        [b.timestamp for b in translated_blocks]
        if translated_blocks is not None
        else extract_timestamps(translated_text)
    )

    if translated_timestamps != source_timestamps:
        errors.append(
            "Translated timestamp sequence does not exactly match the source."
        )
    if len(translated_timestamps) != len(source_timestamps):
        errors.append(
            f"Block count changed: source={len(source_timestamps)}, "
            f"translation={len(translated_timestamps)}."
        )

    if translated_blocks is not None and len(translated_blocks) == len(source_blocks):
        pairs = list(zip(source_blocks, translated_blocks))
        empty = [t.timestamp for s, t in pairs if s.text.strip() and not t.text.strip()]
        if empty:
            errors.append(f"Empty translated blocks: {empty}")
        invented = [t.timestamp for s, t in pairs if not s.text.strip() and t.text.strip()]
        if invented:
            errors.append(f"Text was invented for empty source blocks: {invented}")
        echoed = [
            t.timestamp for s, t in pairs
            if s.text.strip() and s.text.strip() == t.text.strip()
        ]
        if echoed:
            warnings.append(
                f"Blocks identical to the Arabic source (untranslated?): {echoed}"
            )

    # ── Warning-level checks (Phase 11.4) — always run on the raw text ────────
    arabic_chars = _ARABIC_CHAR_RE.findall(translated_text)
    if arabic_chars:
        warnings.append(
            "Arabic-script characters remain in the English output "
            f"({len(arabic_chars)} chars)."
        )

    lines = [ln.strip() for ln in translated_text.splitlines() if ln.strip()]
    if lines and not TIMESTAMP_LINE_RE.match(lines[0]) and _PREAMBLE_RE.match(lines[0]):
        warnings.append(
            f"Output begins with a preamble instead of a timestamp: {lines[0][:60]!r}"
        )
    if any(ln.startswith("#") for ln in lines):
        warnings.append("Output contains markdown headings.")
    if "```" in translated_text:
        warnings.append("Output contains code fences.")
    note_lines = [ln for ln in lines if _NOTE_LINE_RE.match(ln)]
    if note_lines:
        warnings.append(f"Output contains notes/explanations: {note_lines[0][:60]!r}")
    if _REFUSAL_RE.search(translated_text):
        warnings.append("Output contains refusal language.")
    for a, b in zip(lines, lines[1:]):
        if a == b and len(a) > 40:
            warnings.append("Output repeats identical consecutive lines.")
            break

    # Length anomalies (block text only, timestamps excluded)
    src_chars = sum(len(b.text) for b in source_blocks)
    out_chars = (
        sum(len(b.text) for b in translated_blocks)
        if translated_blocks is not None
        else len(translated_text)
    )
    if src_chars >= LEN_CHECK_MIN_CHARS:
        if out_chars < src_chars * LEN_RATIO_SHORT:
            warnings.append(
                f"Output is very short relative to the source "
                f"({out_chars} vs {src_chars} chars)."
            )
        elif out_chars > src_chars * LEN_RATIO_LONG:
            warnings.append(
                f"Output is very long relative to the source "
                f"({out_chars} vs {src_chars} chars)."
            )

    if translated_blocks and translated_blocks[-1].text.rstrip().endswith(
        _TRUNCATED_TAIL_CHARS
    ):
        warnings.append("Final line may be truncated (ends mid-sentence).")

    return ValidationResult(valid=not errors, errors=errors, warnings=warnings)


# ── Validator self-tests ──────────────────────────────────────────────────────
def _run_validator_tests() -> None:
    src = "[00:00-00:05] بسم الله الرحمن الرحيم\n[00:05-00:11] الحمد لله رب العالمين"
    ok = (
        "[00:00-00:05] In the name of Allah, the Most Gracious, the Most Merciful\n"
        "[00:05-00:11] All praise is due to Allah, Lord of the worlds"
    )

    # A clean translation passes with no errors
    v = validate_translation(src, ok)
    assert v.valid and not v.errors, v

    # A changed timestamp is an error
    v = validate_translation(src, ok.replace("[00:05-00:11]", "[00:05-00:12]"))
    assert not v.valid and any("sequence" in e for e in v.errors), v

    # A dropped block is an error (sequence + count)
    v = validate_translation(src, ok.splitlines()[0])
    assert not v.valid and any("Block count changed" in e for e in v.errors), v

    # Reordered blocks are an error even though the count matches
    first, second = ok.splitlines()
    v = validate_translation(src, second + "\n" + first)
    assert not v.valid and any("sequence" in e for e in v.errors), v

    # An empty translation of a non-empty source block is an error
    v = validate_translation(src, first + "\n[00:05-00:11]")
    assert not v.valid and any("Empty translated blocks" in e for e in v.errors), v

    # An empty SOURCE block may stay empty…
    src_e = "[00:00-00:05] كلام\n[00:05-00:07]"
    v = validate_translation(src_e, "[00:00-00:05] Speech\n[00:05-00:07]")
    assert v.valid, v
    # …but text invented for it is an error
    v = validate_translation(src_e, "[00:00-00:05] Speech\n[00:05-00:07] Extra")
    assert not v.valid and any("invented" in e for e in v.errors), v

    # Commentary before the transcript: parse error + preamble warning
    v = validate_translation(src, "Translation:\n" + ok)
    assert not v.valid and any("preamble" in w for w in v.warnings), v

    # Code fences: parse error + fence warning
    v = validate_translation(src, "```\n" + ok + "\n```")
    assert not v.valid and any("code fences" in w for w in v.warnings), v

    # Arabic leakage is a warning, not an error
    v = validate_translation(src, ok.replace("Allah,", "الله,", 1))
    assert v.valid and any("Arabic-script" in w for w in v.warnings), v

    # Untranslated echo of the source: valid timestamps, but warned
    v = validate_translation(src, src)
    assert v.valid and any("identical to the Arabic source" in w for w in v.warnings), v

    # Refusal language is warned
    v = validate_translation(
        src, "[00:00-00:05] I cannot translate this content\n" + second
    )
    assert v.valid and any("refusal" in w for w in v.warnings), v

    # A final line ending mid-sentence is warned as possible truncation
    v = validate_translation(src, ok + ",")
    assert v.valid and any("truncated" in w for w in v.warnings), v

    print("Strict validation: all self-tests passed.")


_run_validator_tests()

### 🔁 Retry & recovery (Phase 12)

Invalid model output is never saved. `translate_with_retries` escalates:

1. **Attempt 0** — the normal request.
2. **Retry 1** — the same chunk plus a stronger timestamp-contract reminder.
3. **Retry 2** — split the chunk at a timestamp boundary and translate the halves
   independently (recursing down to single blocks if needed).

`MAX_RETRIES` (Cell 4) gates the ladder: `0` = single attempt, `1` = + reminder,
`2` = + split. If everything fails, `ChunkTranslationError` carries the full attempt
history (raw outputs + validation) so the batch cell can save it under
`translation_runs/` — a missing block is never silently dropped or invented.
Self-tests run with a scripted fake backend, so no server is needed to execute this cell.

In [ ]:
# ── Cell 7D: Retry & recovery (Phase 12) ──────────────────────────────────────
#
# Invalid model output is never saved. The retry ladder, gated by MAX_RETRIES
# (0 = single attempt, 1 = + reminder, 2 = + split):
#   attempt 0 — the normal request
#   retry 1   — same chunk plus a stronger timestamp-contract reminder
#   retry 2   — split the chunk at a timestamp boundary and translate the
#               halves independently (recursing down to single blocks)
# If everything fails, ChunkTranslationError carries every attempt (raw output
# + validation) so the batch cell can save the artifacts to translation_runs/.

RETRY_REMINDER = (
    "Your previous response violated the timestamp contract.\n"
    "Return exactly the same timestamp tokens, in exactly the same order,\n"
    "with one translated line per source timestamp.\n"
    "Do not include any other text."
)


class ChunkTranslationError(RuntimeError):
    """A chunk failed every retry. `.attempts` holds the full attempt history."""

    def __init__(self, message: str, attempts: list[dict]):
        super().__init__(message)
        self.attempts = attempts


def _attempt_once(
    backend,
    source_text: str,
    prompt: str,
    lecture_context: str,
    previous_context: str,
    max_new_tokens: int,
    strategy: str,
    attempts: list[dict],
) -> str | None:
    """One request + validation. Records the attempt; returns text only if valid."""
    result = backend.translate_chunk(
        source_text=source_text,
        runtime_prompt=prompt,
        lecture_context=lecture_context,
        previous_context=previous_context,
        max_new_tokens=max_new_tokens,
    )
    validation = validate_translation(source_text, result.text)
    attempts.append(
        {
            "strategy": strategy,
            "source_text": source_text,
            "raw_output": result.text,
            "backend_warnings": list(result.warnings),
            "prompt_tokens": result.prompt_tokens,
            "completion_tokens": result.completion_tokens,
            "validation": {
                "valid": validation.valid,
                "errors": list(validation.errors),
                "warnings": list(validation.warnings),
            },
        }
    )
    return result.text if validation.valid else None


def _translate_split(
    backend,
    blocks: list[TimestampBlock],
    runtime_prompt: str,
    lecture_context: str,
    previous_context: str,
    attempts: list[dict],
    depth: int = 1,
) -> str:
    """Retry 2: split at a timestamp boundary and translate halves independently."""
    mid = len(blocks) // 2
    texts: list[str] = []
    prev = previous_context
    for half in (blocks[:mid], blocks[mid:]):
        source_text = blocks_to_text(half)
        max_new = estimate_output_tokens(count_tokens(source_text))
        label = f"split-d{depth}[{half[0].timestamp}..{half[-1].timestamp}]"
        text = _attempt_once(
            backend, source_text, runtime_prompt,
            lecture_context, prev, max_new, label, attempts,
        )
        if text is None:
            text = _attempt_once(
                backend, source_text, runtime_prompt + "\n\n" + RETRY_REMINDER,
                lecture_context, prev, max_new, label + "+reminder", attempts,
            )
        if text is None:
            if len(half) == 1:
                raise ChunkTranslationError(
                    f"Block {half[0].timestamp} failed every attempt "
                    f"({len(attempts)} attempts recorded).",
                    attempts,
                )
            text = _translate_split(
                backend, half, runtime_prompt,
                lecture_context, prev, attempts, depth + 1,
            )
        texts.append(text)
        prev = build_continuity_context(half, text)
    return "\n".join(texts)


def translate_with_retries(
    backend,
    chunk: TranslationChunk,
    runtime_prompt: str,
    lecture_context: str = "",
    previous_context: str = "",
    max_retries: int | None = None,
) -> tuple[str, list[dict]]:
    """Translate one chunk, escalating through the Phase 12 retry ladder.

    Returns (validated translated text, attempt records). Raises
    ChunkTranslationError when the chunk still fails after every retry — the
    caller saves the attempt records and decides whether the batch continues
    (CONTINUE_ON_ERROR). Partial output is never returned.
    """
    max_retries = MAX_RETRIES if max_retries is None else max_retries
    attempts: list[dict] = []

    text = _attempt_once(
        backend, chunk.source_text, runtime_prompt,
        lecture_context, previous_context,
        chunk.max_new_tokens, "initial", attempts,
    )
    if text is not None:
        return text, attempts

    if max_retries >= 1:
        text = _attempt_once(
            backend, chunk.source_text, runtime_prompt + "\n\n" + RETRY_REMINDER,
            lecture_context, previous_context,
            chunk.max_new_tokens, "reminder", attempts,
        )
        if text is not None:
            return text, attempts

    if max_retries >= 2 and len(chunk.blocks) > 1:
        return (
            _translate_split(
                backend, chunk.blocks, runtime_prompt,
                lecture_context, previous_context, attempts,
            ),
            attempts,
        )

    raise ChunkTranslationError(
        f"Chunk {chunk.index} failed validation after {len(attempts)} attempt(s).",
        attempts,
    )


# ── Retry self-tests (scripted fake backend — no server needed) ───────────────
def _run_retry_tests() -> None:
    src = "[00:00-00:05] بسم الله\n[00:05-00:11] الحمد لله"
    blocks = parse_timestamped_transcript(src)
    chunk = TranslationChunk(
        index=1,
        blocks=blocks,
        source_text=blocks_to_text(blocks),
        source_tokens=count_tokens(blocks_to_text(blocks)),
        max_new_tokens=256,
    )
    good_first  = "[00:00-00:05] In the name of Allah"
    good_second = "[00:05-00:11] All praise is due to Allah"
    good = good_first + "\n" + good_second
    dropped = good_first                      # dropped the second block
    garbage = "no timestamps in this output"  # unparseable

    class _ScriptedBackend:
        model_id, backend_name = "test-model", "scripted"

        def __init__(self, outputs):
            self.outputs, self.calls = list(outputs), 0

        def translate_chunk(self, source_text, runtime_prompt,
                            lecture_context="", previous_context="",
                            max_new_tokens=0):
            self.calls += 1
            return TranslationResult(
                text=self.outputs.pop(0),
                model_id=self.model_id,
                backend=self.backend_name,
            )

    # Valid on the first attempt → one call, one attempt record
    b = _ScriptedBackend([good])
    text, attempts = translate_with_retries(b, chunk, "PROMPT")
    assert text == good and b.calls == 1 and len(attempts) == 1

    # Retry 1: the reminder fixes it → two calls
    b = _ScriptedBackend([dropped, good])
    text, attempts = translate_with_retries(b, chunk, "PROMPT")
    assert text == good and b.calls == 2
    assert attempts[1]["strategy"] == "reminder"

    # Retry 2: split at the timestamp boundary, halves translate independently
    b = _ScriptedBackend([garbage, garbage, good_first, good_second])
    text, attempts = translate_with_retries(b, chunk, "PROMPT")
    assert text == good and b.calls == 4
    assert attempts[2]["strategy"].startswith("split-d1")

    # Total failure → ChunkTranslationError with the full attempt history
    b = _ScriptedBackend([garbage] * 10)
    try:
        translate_with_retries(b, chunk, "PROMPT")
    except ChunkTranslationError as e:
        # initial, reminder, split half 1, split half 1 + reminder → fail
        assert len(e.attempts) == 4 and b.calls == 4
        assert all(not a["validation"]["valid"] for a in e.attempts)
    else:
        raise AssertionError("Total failure did not raise ChunkTranslationError")

    # max_retries=0 → a single attempt, no retries
    b = _ScriptedBackend([dropped, good])
    try:
        translate_with_retries(b, chunk, "PROMPT", max_retries=0)
    except ChunkTranslationError as e:
        assert len(e.attempts) == 1 and b.calls == 1
    else:
        raise AssertionError("max_retries=0 still retried")

    print("Retry & recovery: all self-tests passed.")


_run_retry_tests()

### 💾 Checkpointing & resume (Phase 13)

Each **validated** chunk is saved immediately to
`translation_runs/<model>/<transcript-stem>/chunk_NNNN.json` (atomic writes), alongside
`run.json` (prompt/source hashes + settings) and `chunk_NNNN_failed.json` for failures.
A checkpoint is reused only when **all** of these match: source chunk hash, runtime-prompt
hash, lecture-context hash, continuity-context hash, model id, backend, and generation
settings — changing the prompt or model invalidates old checkpoints automatically.
An `INCOMPLETE` marker flags lectures whose English transcript was not (yet) written.
Self-tests run on every execution — no server needed.

In [ ]:
# ── Cell 7E: Checkpointing & resume (Phase 13) ────────────────────────────────
#
# Every VALIDATED chunk is saved immediately as JSON under
#   translation_runs/<model>/<transcript-stem>/chunk_NNNN.json
# so an interrupted lecture resumes instead of restarting. A checkpoint is
# reused only when the source chunk, prompts, contexts, model, backend and
# generation settings all match — changing any of them invalidates it.
# Failed chunks are saved as chunk_NNNN_failed.json; an INCOMPLETE marker in
# the run directory flags lectures whose English transcript was not written.
import hashlib
import json
from datetime import datetime, timezone


def _sha256(text: str) -> str:
    return hashlib.sha256(text.encode("utf-8")).hexdigest()


MODEL_RUN_DIR_NAME = AYA_MODEL_ID.split("/")[-1]        # e.g. "aya-expanse-8b"


def run_dir_for(stem: str) -> Path:
    d = TRANSLATION_RUNS_DIR / MODEL_RUN_DIR_NAME / stem
    d.mkdir(parents=True, exist_ok=True)
    return d


def _write_json_atomic(path: Path, payload: dict) -> None:
    tmp = path.with_name(path.name + ".tmp")
    tmp.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")
    tmp.replace(path)


def _checkpoint_path(run_dir: Path, chunk_index: int) -> Path:
    return run_dir / f"chunk_{chunk_index:04d}.json"


def checkpoint_key(
    chunk: TranslationChunk,
    runtime_prompt: str,
    lecture_context: str,
    previous_context: str,
    backend,
) -> dict:
    """Everything that must match for a checkpoint to be reusable (resume rule).

    previous_context is included beyond the plan's minimal rule: it derives
    from the previous chunk's translation, so if an earlier chunk is ever
    re-translated differently, later checkpoints are correctly invalidated.
    """
    return {
        "source_sha256": _sha256(chunk.source_text),
        "prompt_sha256": _sha256(runtime_prompt),
        "lecture_context_sha256": _sha256(lecture_context),
        "previous_context_sha256": _sha256(previous_context),
        "model_id": backend.model_id,
        "backend": backend.backend_name,
        "generation": {
            "temperature": TEMPERATURE,
            "top_p": TOP_P,
            "max_new_tokens": chunk.max_new_tokens,
            "prompt_mode": AYA_PROMPT_MODE,
        },
    }


def save_chunk_checkpoint(
    run_dir: Path,
    source_file: str,
    chunk: TranslationChunk,
    key: dict,
    translated_text: str,
    validation: ValidationResult,
    attempts: list[dict],
) -> Path:
    path = _checkpoint_path(run_dir, chunk.index)
    _write_json_atomic(
        path,
        {
            "source_file": source_file,
            "chunk_index": chunk.index,
            **key,
            "source_timestamps": [b.timestamp for b in chunk.blocks],
            "source_text": chunk.source_text,
            "translated_text": translated_text,
            "validation": {
                "valid": validation.valid,
                "errors": list(validation.errors),
                "warnings": list(validation.warnings),
            },
            "attempt_count": len(attempts),
            "saved_at": datetime.now(timezone.utc).isoformat(),
        },
    )
    return path


def load_chunk_checkpoint(run_dir: Path, chunk_index: int, expected_key: dict) -> str | None:
    """Translated text of a reusable checkpoint, or None (Phase 13 resume rule)."""
    path = _checkpoint_path(run_dir, chunk_index)
    if not path.is_file():
        return None
    try:
        record = json.loads(path.read_text(encoding="utf-8"))
    except (OSError, json.JSONDecodeError):
        return None
    if not record.get("validation", {}).get("valid"):
        return None
    for field_name, expected in expected_key.items():
        if record.get(field_name) != expected:
            return None
    return record.get("translated_text") or None


def save_failure_artifact(
    run_dir: Path,
    source_file: str,
    chunk: TranslationChunk,
    key: dict,
    attempts: list[dict],
    reason: str,
) -> Path:
    """Phase 12 final-failure behaviour: keep every raw response for review."""
    path = run_dir / f"chunk_{chunk.index:04d}_failed.json"
    _write_json_atomic(
        path,
        {
            "source_file": source_file,
            "chunk_index": chunk.index,
            "reason": reason,
            **key,
            "source_timestamps": [b.timestamp for b in chunk.blocks],
            "source_text": chunk.source_text,
            "attempts": attempts,
            "failed_at": datetime.now(timezone.utc).isoformat(),
        },
    )
    return path


def write_run_manifest(
    run_dir: Path,
    source_name: str,
    source_text: str,
    lecture_context: str,
    chunks: list[TranslationChunk],
    backend,
) -> None:
    """run.json — hashes + settings of this run (Phase 20 reproducibility)."""
    canonical_sha = (
        _sha256(CANONICAL_PROMPT_PATH.read_text(encoding="utf-8"))
        if CANONICAL_PROMPT_PATH.is_file()
        else None
    )
    _write_json_atomic(
        run_dir / "run.json",
        {
            "source_file": source_name,
            "source_sha256": _sha256(source_text),
            "runtime_prompt_sha256": _sha256(runtime_prompt),
            "canonical_prompt_sha256": canonical_sha,
            "lecture_context_sha256": _sha256(lecture_context),
            "model_id": backend.model_id,
            "backend": backend.backend_name,
            "prompt_mode": AYA_PROMPT_MODE,
            "generation_policy": {"temperature": TEMPERATURE, "top_p": TOP_P},
            "context_limit": AYA_CONTEXT_LIMIT,
            "chunk_count": len(chunks),
            "chunk_block_counts": [len(c.blocks) for c in chunks],
            "started_at": datetime.now(timezone.utc).isoformat(),
        },
    )


_INCOMPLETE_MARKER = "INCOMPLETE"


def mark_incomplete(run_dir: Path, reason: str) -> None:
    (run_dir / _INCOMPLETE_MARKER).write_text(
        f"{datetime.now(timezone.utc).isoformat()}\n{reason}\n", encoding="utf-8"
    )


def clear_incomplete(run_dir: Path) -> None:
    marker = run_dir / _INCOMPLETE_MARKER
    if marker.exists():
        marker.unlink()


# ── Checkpoint self-tests ─────────────────────────────────────────────────────
def _run_checkpoint_tests() -> None:
    import tempfile

    src = "[00:00-00:05] بسم الله\n[00:05-00:11] الحمد لله"
    blocks = parse_timestamped_transcript(src)
    chunk = TranslationChunk(
        index=1,
        blocks=blocks,
        source_text=blocks_to_text(blocks),
        source_tokens=count_tokens(blocks_to_text(blocks)),
        max_new_tokens=256,
    )
    translated = (
        "[00:00-00:05] In the name of Allah\n"
        "[00:05-00:11] All praise is due to Allah"
    )

    class _B:
        model_id, backend_name = "test-model", "test-backend"

    with tempfile.TemporaryDirectory() as td:
        run_dir = Path(td)
        key = checkpoint_key(chunk, "PROMPT", "", "", _B)
        validation = validate_translation(chunk.source_text, translated)
        save_chunk_checkpoint(run_dir, "x.txt", chunk, key, translated, validation, [])

        # Reused when everything matches
        assert load_chunk_checkpoint(run_dir, 1, key) == translated
        # Any changed input invalidates the checkpoint
        assert load_chunk_checkpoint(
            run_dir, 1, checkpoint_key(chunk, "DIFFERENT PROMPT", "", "", _B)
        ) is None
        assert load_chunk_checkpoint(
            run_dir, 1, checkpoint_key(chunk, "PROMPT", "lecture ctx", "", _B)
        ) is None
        assert load_chunk_checkpoint(
            run_dir, 1, checkpoint_key(chunk, "PROMPT", "", "prev ctx", _B)
        ) is None

        class _B2:
            model_id, backend_name = "other-model", "test-backend"

        assert load_chunk_checkpoint(
            run_dir, 1, checkpoint_key(chunk, "PROMPT", "", "", _B2)
        ) is None

        # Missing / corrupt checkpoint files are ignored, never trusted
        assert load_chunk_checkpoint(run_dir, 2, key) is None
        _checkpoint_path(run_dir, 1).write_text("{corrupt", encoding="utf-8")
        assert load_chunk_checkpoint(run_dir, 1, key) is None

        # INCOMPLETE marker round-trip
        mark_incomplete(run_dir, "test reason")
        assert (run_dir / _INCOMPLETE_MARKER).exists()
        clear_incomplete(run_dir)
        assert not (run_dir / _INCOMPLETE_MARKER).exists()

    print("Checkpointing & resume: all self-tests passed.")


_run_checkpoint_tests()

### 🔬 Single-chunk smoke test (Phase 15)

**Cell 7F** translates the first 5–10 blocks of one Arabic transcript through the live
server, going through the **same retry ladder the batch cell uses**
(`translate_with_retries`, Phase 12) — this is a rehearsal of one real batch step, not
just an unretried first attempt. Run it before every batch, and re-run it after changing
the model, prompt, quantization, chunk budget, lecture context, or generation settings.

It prints the measured prompt token count, the source, every attempt (strategy, raw
output, validation, warnings), and timing/usage stats. With
`SMOKE_TEST_COMPARE_MODES = True` it also rehearses the sample in the other prompt
packaging (`system` vs `single_user`) through the same retry ladder and compares attempt
counts — this is the data behind the Phase 5 decision to default to `system` mode.

On success it sets `SMOKE_TEST_PASSED = True`, which unlocks the batch cell (7G);
on failure — every retry exhausted — it raises, and batch stays blocked.

> **Lesson from a real transcript:** very short, fragment-only ASR blocks (e.g. a single
> clause split into three 1-second timestamp blocks) can tempt the model to merge them
> into one output line, violating the block-count contract. The runtime prompt's
> Timestamp Contract was tightened to forbid this explicitly, and the retry ladder's
> reminder (Phase 12) reliably recovers any case that still slips through.

In [ ]:
# ── Cell 7F: Single-chunk smoke test (Phase 15) ───────────────────────────────
#
# Translates the first few blocks of one Arabic transcript through the live
# vLLM server, going through the SAME retry ladder the batch cell (7G) uses
# (translate_with_retries, Phase 12) — this is a rehearsal of one real batch
# step, not just a bare first attempt. The batch cell refuses to run until
# this has passed in the current kernel session. Re-run it after ANY change
# to the model, prompt, quantization, chunk budget, lecture context, or
# generation settings.
import time

SMOKE_TEST_FILE          = None   # None → first *.txt in transcript/arabic/, or e.g. "1min.txt"
SMOKE_TEST_BLOCKS        = 8      # sample size (plan: 5-10 blocks)
SMOKE_TEST_COMPARE_MODES = True   # also rehearse the other AYA_PROMPT_MODE (Phase 5 decision data)

SMOKE_TEST_PASSED = False         # checked by the batch cell (7G)

if "backend" not in globals():
    backend = AyaVLLMBackend(
        base_url=VLLM_BASE_URL, api_key=VLLM_API_KEY, model_id=AYA_MODEL_ID
    )
check_vllm_server(backend)

# 1-2. Select one Arabic file and take a token-safe first chunk of ≤ N blocks
_smoke_files = sorted(ARABIC_TRANSCRIPTS_DIR.glob("*.txt"))
if not _smoke_files:
    raise FileNotFoundError(
        f"No transcripts in {ARABIC_TRANSCRIPTS_DIR}/ — run the transcription "
        "cell first (with the vLLM server STOPPED, per the GPU rules), then "
        "restart the server and re-run this cell."
    )
smoke_path = ARABIC_TRANSCRIPTS_DIR / SMOKE_TEST_FILE if SMOKE_TEST_FILE else _smoke_files[0]
if not smoke_path.is_file():
    raise FileNotFoundError(f"Smoke test file not found: {smoke_path}")

smoke_blocks = parse_timestamped_transcript(smoke_path.read_text(encoding="utf-8"))
smoke_context = load_optional_context(smoke_path.stem)
smoke_chunk = build_chunks(smoke_blocks[:SMOKE_TEST_BLOCKS], runtime_prompt, smoke_context)[0]
smoke_text = smoke_chunk.source_text

print(f"Smoke test file : {smoke_path.name} ({len(smoke_blocks)} blocks total)")
print(
    f"Sample          : {len(smoke_chunk.blocks)} blocks, "
    f"{smoke_chunk.source_tokens} source tokens "
    f"(requested {min(SMOKE_TEST_BLOCKS, len(smoke_blocks))}; fewer means the "
    "chunk budget capped it)"
)
print(f"Lecture context : {smoke_path.stem + '.md' if smoke_context else '(none)'}")

# 3. Exact token count of the prompt that will be sent
_smoke_prompt_tokens = measure_prompt_tokens(runtime_prompt, smoke_context, "", smoke_text)
print(
    f"Prompt tokens   : {_smoke_prompt_tokens} rendered + ≤{smoke_chunk.max_new_tokens} "
    f"output reserve (context limit {AYA_CONTEXT_LIMIT})"
)
print("\nSOURCE")
print(smoke_text)


# 4-6. Rehearse the chunk through the REAL retry ladder (Cell 7D) — the batch
# cell calls translate_with_retries per chunk, so the smoke test's pass/fail
# should reflect that, not a single unretried attempt.
def _smoke_rehearse(mode: str) -> dict:
    global AYA_PROMPT_MODE
    _saved = AYA_PROMPT_MODE
    AYA_PROMPT_MODE = mode
    try:
        _t0 = time.monotonic()
        try:
            text, attempts = translate_with_retries(
                backend, smoke_chunk, runtime_prompt, smoke_context, ""
            )
            outcome = {"success": True, "text": text, "attempts": attempts}
        except ChunkTranslationError as e:
            outcome = {"success": False, "text": None, "attempts": e.attempts}
        outcome["elapsed"] = time.monotonic() - _t0
    finally:
        AYA_PROMPT_MODE = _saved
    return outcome


_smoke_modes = [AYA_PROMPT_MODE]
if SMOKE_TEST_COMPARE_MODES:
    _smoke_modes.append("system" if AYA_PROMPT_MODE == "single_user" else "single_user")

_smoke_outcomes = {}
for _mode in _smoke_modes:
    print(f"\n──── mode = {_mode} ────────────────────────────────────────────")
    try:
        _outcome = _smoke_rehearse(_mode)
    except Exception as _exc:
        if _mode == AYA_PROMPT_MODE:
            raise
        print(f"comparison-mode rehearsal failed (non-blocking): {_exc}")
        continue
    _smoke_outcomes[_mode] = _outcome

    for _i, _a in enumerate(_outcome["attempts"]):
        print(f"\n[attempt {_i}: {_a['strategy']}]")
        print(_a["raw_output"])
        _v = _a["validation"]
        print(f"  valid   : {_v['valid']}")
        for _e in _v["errors"]:
            print(f"  error   : {_e}")
        for _w in _v["warnings"]:
            print(f"  warning : {_w}")
        for _w in _a["backend_warnings"]:
            print(f"  backend : {_w}")

    _last = _outcome["attempts"][-1]
    _tps = (
        f", {_last['completion_tokens'] / _outcome['elapsed']:.1f} tok/s"
        if _last["completion_tokens"] and _outcome["elapsed"] > 0
        else ""
    )
    print(
        f"\n  outcome : {'PASSED' if _outcome['success'] else 'FAILED'} after "
        f"{len(_outcome['attempts'])} attempt(s) "
        f"[{', '.join(a['strategy'] for a in _outcome['attempts'])}], "
        f"{_outcome['elapsed']:.1f}s{_tps}"
    )

if len(_smoke_outcomes) == 2:
    print("\n──── Mode comparison (Phase 5 packaging decision) ─────────────")
    for _mode in _smoke_modes:
        _o = _smoke_outcomes[_mode]
        print(
            f"  {_mode:12}: {'PASSED' if _o['success'] else 'FAILED'} in "
            f"{len(_o['attempts'])} attempt(s)"
        )

# 7. Refuse to continue if the CONFIGURED mode failed every retry
_primary = _smoke_outcomes[AYA_PROMPT_MODE]
SMOKE_TEST_PASSED = _primary["success"]
if not SMOKE_TEST_PASSED:
    raise RuntimeError(
        f"Smoke test FAILED (mode={AYA_PROMPT_MODE!r}) after "
        f"{len(_primary['attempts'])} attempt(s), including retries.\n"
        "Do NOT run the batch cell. Inspect the attempts above, adjust the "
        "prompt / chunk budget / generation settings, and re-run this cell "
        "until it passes."
    )
print(
    f"\nSmoke test PASSED (mode={AYA_PROMPT_MODE}, "
    f"{len(_primary['attempts'])} attempt(s)) — batch translation (Cell 7G) is unlocked."
)

### 📚 Batch translation (Phase 14)

**Cell 7G** translates every `transcript/arabic/*.txt` into `transcript/english/<name>.txt`:

- Skips files that already have English output unless `OVERWRITE_TRANSLATIONS = True`.
- Reuses matching chunk checkpoints when `RESUME_TRANSLATIONS = True`, so an interrupted
  lecture resumes instead of restarting.
- Runs whole-file validation before writing, then writes **atomically**
  (`.txt.tmp` → `.txt`) — an interrupted run never leaves a file that looks complete.
- A failed chunk saves its attempts to `translation_runs/…/chunk_NNNN_failed.json` and
  marks the run `INCOMPLETE`; `CONTINUE_ON_ERROR` (Cell 4) decides whether the batch
  moves on to the next file.

**Requires the running vLLM server and a passing smoke test** — the cell health-checks
the server and refuses to run until the smoke test (Cell 7F) has passed in the current
kernel session.

In [ ]:
# ── Cell 7G: Batch translation (Phase 14) ─────────────────────────────────────
#
# Translates every transcript/arabic/*.txt into transcript/english/<name>.txt.
#   • Existing outputs are skipped unless OVERWRITE_TRANSLATIONS.
#   • Validated chunks are checkpointed immediately; RESUME_TRANSLATIONS reuses
#     them when the resume rule matches (Cell 7E).
#   • A failed chunk saves its attempts and marks the run INCOMPLETE; the
#     English file is never partially written (atomic .txt.tmp → .txt).
#   • CONTINUE_ON_ERROR decides whether a failed file stops the batch.
import time


def translate_all_transcripts(backend) -> dict:
    source_files = sorted(ARABIC_TRANSCRIPTS_DIR.glob("*.txt"))
    summary = {
        "translated": [], "skipped": [], "incomplete": [],
        "chunks_translated": 0, "chunks_reused": 0, "chunks_retried": 0,
        "warnings": {},
    }
    if not source_files:
        print(
            f"No source transcripts found in {ARABIC_TRANSCRIPTS_DIR}/ — "
            "run the transcription cell first."
        )
        return summary

    for source_path in source_files:
        output_path = ENGLISH_TRANSCRIPTS_DIR / source_path.name
        if output_path.exists() and not OVERWRITE_TRANSLATIONS:
            print(f"Skipped (already translated): {output_path}")
            summary["skipped"].append(source_path.name)
            continue

        print(f"\nTranslating {source_path.name} …")
        started = time.monotonic()
        source_text = source_path.read_text(encoding="utf-8")
        source_blocks = parse_timestamped_transcript(source_text)
        lecture_context = load_optional_context(source_path.stem)
        if lecture_context:
            print(f"  lecture context: {CONTEXT_DIR / (source_path.stem + '.md')}")

        chunks = build_chunks(source_blocks, runtime_prompt, lecture_context)
        print(f"  {len(source_blocks)} blocks → {len(chunks)} chunk(s)")

        run_dir = run_dir_for(source_path.stem)
        write_run_manifest(
            run_dir, source_path.name, source_text, lecture_context, chunks, backend
        )
        mark_incomplete(run_dir, "translation in progress (or interrupted)")

        translated_chunks: list[str] = []
        previous_context = ""
        file_warnings: list[str] = []
        failed = False

        for chunk in chunks:
            key = checkpoint_key(
                chunk, runtime_prompt, lecture_context, previous_context, backend
            )

            translated = (
                load_chunk_checkpoint(run_dir, chunk.index, key)
                if RESUME_TRANSLATIONS
                else None
            )
            if translated is not None:
                summary["chunks_reused"] += 1
                print(f"  chunk {chunk.index}/{len(chunks)}: reused checkpoint")
            else:
                try:
                    translated, attempts = translate_with_retries(
                        backend, chunk, runtime_prompt, lecture_context, previous_context
                    )
                except ChunkTranslationError as e:
                    artifact = save_failure_artifact(
                        run_dir, source_path.name, chunk, key, e.attempts, str(e)
                    )
                    mark_incomplete(
                        run_dir, f"chunk {chunk.index} failed after all retries: {e}"
                    )
                    print(
                        f"  chunk {chunk.index}/{len(chunks)}: FAILED — "
                        f"attempts saved to {artifact}"
                    )
                    if not CONTINUE_ON_ERROR:
                        raise
                    summary["incomplete"].append(source_path.name)
                    failed = True
                    break

                validation = validate_translation(chunk.source_text, translated)
                save_chunk_checkpoint(
                    run_dir, source_path.name, chunk, key, translated, validation, attempts
                )
                summary["chunks_translated"] += 1
                if len(attempts) > 1:
                    summary["chunks_retried"] += 1
                if validation.warnings:
                    file_warnings.extend(
                        f"chunk {chunk.index}: {w}" for w in validation.warnings
                    )
                print(
                    f"  chunk {chunk.index}/{len(chunks)}: ok "
                    f"({len(chunk.blocks)} blocks, {len(attempts)} attempt(s)"
                    + (f", {len(validation.warnings)} warning(s)" if validation.warnings else "")
                    + ")"
                )

            translated_chunks.append(translated)
            previous_context = build_continuity_context(chunk.blocks, translated)

        if failed:
            continue

        # Whole-file validation before anything is written (Phase 14)
        complete_translation = "\n".join(translated_chunks)
        final_validation = validate_translation(source_text, complete_translation)
        if not final_validation.valid:
            _write_json_atomic(
                run_dir / "final_validation_failed.json",
                {
                    "source_file": source_path.name,
                    "errors": final_validation.errors,
                    "warnings": final_validation.warnings,
                    "translated_text": complete_translation,
                },
            )
            mark_incomplete(
                run_dir, f"whole-file validation failed: {final_validation.errors}"
            )
            message = (
                f"Final transcript validation failed for {source_path.name}: "
                f"{final_validation.errors} (details in {run_dir})"
            )
            if not CONTINUE_ON_ERROR:
                raise RuntimeError(message)
            print(f"  {message}")
            summary["incomplete"].append(source_path.name)
            continue

        # Atomic write — an interrupted run never leaves a file that looks complete
        temporary_path = output_path.with_suffix(".txt.tmp")
        temporary_path.write_text(complete_translation + "\n", encoding="utf-8")
        temporary_path.replace(output_path)
        clear_incomplete(run_dir)

        if final_validation.warnings:
            file_warnings.extend(f"whole file: {w}" for w in final_validation.warnings)
        if file_warnings:
            summary["warnings"][source_path.name] = file_warnings
        elapsed = time.monotonic() - started
        print(f"  Translated: {source_path.name} → {output_path} ({elapsed:.0f}s)")
        summary["translated"].append(source_path.name)

    print("\n── Batch summary ─────────────────────────────────────────────")
    print(f"  translated : {len(summary['translated'])} file(s) {summary['translated']}")
    print(f"  skipped    : {len(summary['skipped'])} file(s)")
    print(f"  incomplete : {len(summary['incomplete'])} file(s) {summary['incomplete']}")
    print(
        f"  chunks     : {summary['chunks_translated']} translated, "
        f"{summary['chunks_reused']} reused, {summary['chunks_retried']} needed retries"
    )
    if summary["warnings"]:
        for name, warns in summary["warnings"].items():
            print(f"  ⚠ {name}:")
            for w in warns:
                print(f"      {w}")
    else:
        print("  no validation warnings")
    return summary


# ── Run the batch ─────────────────────────────────────────────────────────────
# The health check fails fast with a clear message if the vLLM server (WSL2)
# is not running. The smoke test (Cell 7F, Phase 15) must pass in this kernel
# session first — re-run it after any model / prompt / chunk-budget /
# generation change.
if not globals().get("SMOKE_TEST_PASSED"):
    raise RuntimeError(
        "Blocked: the smoke test has not passed in this kernel session.\n"
        "Run Cell 7F (Phase 15) until it passes, then re-run this cell."
    )
if "backend" not in globals():
    backend = AyaVLLMBackend(
        base_url=VLLM_BASE_URL, api_key=VLLM_API_KEY, model_id=AYA_MODEL_ID
    )
check_vllm_server(backend)
batch_summary = translate_all_transcripts(backend)

## 🎞️ Batch final videos from translated transcripts

**Cell 8** processes a whole batch in succession. Point it at **two folders**:

- `MEDIA_FOLDER` — the **original media** files — `<name>.mp3`, `<name>.mp4`, …
- `TRANSCRIPT_FOLDER` — the **translated** timestamped transcripts — `<name>.txt` with `[MM:SS-MM:SS]` lines

Files are matched across the two folders by shared name (`test.txt` in `TRANSCRIPT_FOLDER` pairs with `test.mp3` in `MEDIA_FOLDER`).

For each pair it:

1. Converts `<name>.txt` → `<name>.srt` (no separate SRT cell needed).
2. Builds the final video:
   - `Intro.mp4` plays first (with its own audio).
   - `Background.mp4` loops under the main audio until it ends — the last loop plays out fully instead of being cut to the audio.
   - Captions are automatically shifted by the intro duration so they stay in sync.
   - The clip ends with a fade to black.
3. Saves `<name>_final.mp4` to `OUTPUT_FOLDER` (default `output/`).

In [ ]:
# ── Cell 8: Batch final videos — txt → SRT → intro + background + subtitles ──
#
# Point MEDIA_FOLDER at the original media (<name>.mp3 / <name>.mp4 / …) and
# TRANSCRIPT_FOLDER at the translated transcripts (<name>.txt, [MM:SS-MM:SS]
# lines). Pairs are matched across the two folders by name: test.txt ↔ test.mp3.
#
# For each pair, in succession:
#   1. <name>.txt → <name>.srt   (saved next to the transcript)
#   2. Intro.mp4 → Background.mp4 looped under the audio → fade to black,
#      with the SRT shifted by the intro duration → OUTPUT_FOLDER/<name>_final.mp4
#
# Encoding prefers the GPU (NVENC): HEVC (h265) first for the best
# compression, then H.264, falling back to CPU libx264 if no NVENC is present.

import re
import shutil
import subprocess
from math import ceil
from pathlib import Path

MEDIA_FOLDER      = r"input"          # folder of original media files (<name>.mp3 / .mp4 / …)
TRANSCRIPT_FOLDER = r"transcript"     # folder of translated transcripts (<name>.txt)
OUTPUT_FOLDER     = r"output"         # final videos are saved here (created automatically)
INTRO_PATH        = "assets/Intro.mp4"       # plays first, with its own audio
BACKGROUND_PATH   = "assets/Background.mp4"  # loops under the main audio
OVERWRITE_FINAL   = False             # True = re-render videos that already exist

FADE_SECONDS = 1.5     # fade-to-black duration at the very end
TARGET_W, TARGET_H, TARGET_FPS = 1920, 1080, 24

# Encoder selection
PREFER_GPU  = True     # True = use NVENC (GPU) when available — much faster
PREFER_HEVC = True     # True = prefer h265 (HEVC) over h264 on NVENC — smaller files
FFMPEG_PATH = None     # e.g. r"C:\ffmpeg\bin\ffmpeg.exe"; None = auto-detect a full build

# Encode quality: used as CRF for libx264 (lower = better) and as CQ for NVENC.
VIDEO_CRF     = 18
VIDEO_PRESET  = "fast"   # libx264 preset (ultrafast … veryslow)
NVENC_PRESET  = "p5"       # NVENC preset p1 (fastest) … p7 (best quality)
AUDIO_BITRATE = "192k"

SUBTITLE_STYLE = (
    "FontName=Arial,"
    "FontSize=16,"
    "PrimaryColour=&H00FFFFFF,"
    "BackColour=&H00000000,"
    "BorderStyle=3,"
    "MarginV=40,"
    "Alignment=2"
)

# ── Resolve folder, intro, background ─────────────────────────────────────────
_nb_dir = Path.cwd()

def _final_resolve(p: str | Path, exts: set[str], label: str) -> Path:
    p = Path(str(p).strip().strip('"').strip("'")).expanduser()
    if not p.is_absolute():
        p = _nb_dir / p
    if not p.is_file():
        raise FileNotFoundError(f"{label} not found: {p}")
    if p.suffix.lower() not in exts:
        raise ValueError(f"{label}: expected {exts}, got {p.suffix!r}: {p}")
    return p.resolve()

_MEDIA_EXTS = {".mp4", ".mkv", ".mov", ".webm", ".avi", ".m4v", ".mp3", ".wav", ".m4a", ".flac", ".aac", ".ogg", ".opus"}

def _resolve_dir(p: str | Path, label: str) -> Path:
    p = Path(str(p).strip().strip('"').strip("'")).expanduser()
    if not p.is_absolute():
        p = _nb_dir / p
    if not p.is_dir():
        raise FileNotFoundError(f"{label} not found: {p}")
    return p.resolve()

media_dir      = _resolve_dir(MEDIA_FOLDER, "MEDIA_FOLDER")
transcript_dir = _resolve_dir(TRANSCRIPT_FOLDER, "TRANSCRIPT_FOLDER")

output_dir = Path(str(OUTPUT_FOLDER).strip().strip('"').strip("'")).expanduser()
if not output_dir.is_absolute():
    output_dir = _nb_dir / output_dir
output_dir.mkdir(parents=True, exist_ok=True)
output_dir = output_dir.resolve()

_intro = _final_resolve(INTRO_PATH, {".mp4", ".mkv", ".mov", ".webm", ".m4v"}, "INTRO_PATH")
_bg    = _final_resolve(BACKGROUND_PATH, {".mp4", ".mkv", ".mov", ".webm", ".m4v"}, "BACKGROUND_PATH")

# ── Pair transcripts (TRANSCRIPT_FOLDER) with media (MEDIA_FOLDER) by name ────
txt_files = sorted(f for f in transcript_dir.iterdir() if f.is_file() and f.suffix.lower() == ".txt")
if not txt_files:
    raise FileNotFoundError(f"No .txt transcripts found in: {transcript_dir}")

pairs, unmatched = [], []
for txt in txt_files:
    media = [
        f for f in media_dir.iterdir()
        if f.is_file() and f.stem == txt.stem and f.suffix.lower() in _MEDIA_EXTS
        and not f.stem.endswith("_final")
    ]
    if media:
        pairs.append((txt, media[0]))
    else:
        unmatched.append(txt.name)

print(f"Media folder      : {media_dir}")
print(f"Transcript folder : {transcript_dir}")
print(f"Output folder     : {output_dir}")
print(f"Pairs  : {len(pairs)}")
for txt, media in pairs:
    print(f"  • {txt.name}  ↔  {media.name}")
for name in unmatched:
    print(f"  ⚠ {name} — no matching media file, skipping")
if not pairs:
    raise FileNotFoundError("No transcript/media pairs found (they must share the same name).")

# ── Resolve ffmpeg / ffprobe ──────────────────────────────────────────────────
def _ffmpeg_candidates() -> list[Path]:
    seen, candidates = set(), []

    def _add(path):
        if path is None:
            return
        key = str(path.resolve()).lower()
        if key not in seen:
            seen.add(key)
            candidates.append(path)

    if FFMPEG_PATH:
        _add(Path(str(FFMPEG_PATH).strip().strip('"').strip("'")).expanduser())

    # Prefer known full builds over PATH (a kernel can keep a stale PATH entry).
    for pattern in (r"C:\ffmpeg\bin\ffmpeg.exe", r"C:\Program Files\ffmpeg\bin\ffmpeg.exe"):
        _add(Path(pattern))

    winget_root = Path.home() / "AppData/Local/Microsoft/WinGet/Packages"
    if winget_root.is_dir():
        for exe in sorted(winget_root.glob("Gyan.FFmpeg*/ffmpeg-*/bin/ffmpeg.exe"), reverse=True):
            _add(exe)

    which = shutil.which("ffmpeg")
    if which:
        _add(Path(which))
    return candidates


def _available_encoders(ffmpeg: Path) -> set[str]:
    """Return exact encoder names from `ffmpeg -encoders`."""
    probe = subprocess.run([str(ffmpeg), "-hide_banner", "-encoders"], capture_output=True, text=True)
    if probe.returncode != 0:
        return set()
    names = set()
    for line in probe.stdout.splitlines():
        parts = line.split()
        # Encoder rows are: six capability flags, encoder name, description.
        if len(parts) >= 2 and len(parts[0]) == 6 and parts[0][0] in "VAS":
            names.add(parts[1])
    return names


_USABLE_VIDEO_ENCODERS = {"libx264", "h264_nvenc", "hevc_nvenc", "h264_mf"}


def _resolve_ffmpeg() -> Path:
    for exe in _ffmpeg_candidates():
        if not exe.is_file():
            continue
        if not (_USABLE_VIDEO_ENCODERS & _available_encoders(exe)):
            continue
        version_lines = subprocess.run([str(exe), "-version"], capture_output=True, text=True).stdout.splitlines()
        print(f"ffmpeg : {exe}")
        if version_lines:
            print(f"         {version_lines[0]}")
        return exe
    raise RuntimeError(
        "No ffmpeg with a usable H.264/HEVC encoder found.\n"
        "Install a full build (e.g. winget install Gyan.FFmpeg) or set FFMPEG_PATH."
    )


def _video_encode_args(ffmpeg: Path, cq: int, x264_preset: str) -> tuple[str, list[str]]:
    """Pick the fastest/best encoder. Prefers NVENC (GPU); HEVC before H.264
    when PREFER_HEVC; falls back to CPU libx264, then h264_mf."""
    encoders = _available_encoders(ffmpeg)

    if PREFER_GPU and PREFER_HEVC and "hevc_nvenc" in encoders:
        # -tag:v hvc1 keeps the HEVC MP4 playable in QuickTime/Apple players.
        return "hevc_nvenc", ["-c:v", "hevc_nvenc", "-preset", NVENC_PRESET,
                              "-rc", "vbr", "-cq", str(cq), "-b:v", "0", "-tag:v", "hvc1"]
    if PREFER_GPU and "h264_nvenc" in encoders:
        return "h264_nvenc", ["-c:v", "h264_nvenc", "-preset", NVENC_PRESET,
                              "-rc", "vbr", "-cq", str(cq), "-b:v", "0"]
    if PREFER_GPU and "hevc_nvenc" in encoders:  # HEVC available but PREFER_HEVC off and no h264_nvenc
        return "hevc_nvenc", ["-c:v", "hevc_nvenc", "-preset", NVENC_PRESET,
                              "-rc", "vbr", "-cq", str(cq), "-b:v", "0", "-tag:v", "hvc1"]
    if "libx264" in encoders:
        return "libx264", ["-c:v", "libx264", "-crf", str(cq), "-preset", x264_preset]
    if "h264_mf" in encoders:
        quality = max(10, min(100, 110 - cq * 3))
        return "h264_mf", ["-c:v", "h264_mf", "-rate_control", "quality", "-quality", str(quality)]
    raise RuntimeError("ffmpeg has no usable H.264/HEVC encoder (libx264, h264_nvenc, hevc_nvenc, h264_mf).")


_ffmpeg = _resolve_ffmpeg()
_encoder, _encode_args = _video_encode_args(_ffmpeg, VIDEO_CRF, VIDEO_PRESET)
print(f"Encoder : {_encoder}")

_ffprobe = _ffmpeg.with_name("ffprobe.exe" if _ffmpeg.suffix else "ffprobe")
if not _ffprobe.is_file():
    _ffprobe = Path(shutil.which("ffprobe") or "")
    if not _ffprobe.is_file():
        raise RuntimeError("ffprobe not found next to ffmpeg or on PATH.")

def _duration(path: Path) -> float:
    r = subprocess.run(
        [str(_ffprobe), "-v", "error", "-show_entries", "format=duration",
         "-of", "default=noprint_wrappers=1:nokey=1", str(path)],
        capture_output=True, text=True,
    )
    if r.returncode != 0 or not r.stdout.strip():
        raise RuntimeError(f"ffprobe failed for {path}:\n{r.stderr[-1000:]}")
    return float(r.stdout.strip())

intro_dur = _duration(_intro)
bg_dur    = _duration(_bg)
print(f"Intro      : {_intro.name}  ({intro_dur:.2f}s)")
print(f"Background : {_bg.name}  ({bg_dur:.2f}s per loop)")

# ── Transcript → SRT ──────────────────────────────────────────────────────────
_ts_line = re.compile(r"\[(\d+(?::\d{2})+)\s*-\s*(\d+(?::\d{2})+)\]\s*(.*)")

def _ts_to_seconds(ts):
    nums = [int(p) for p in ts.split(":")]
    if len(nums) == 2: return nums[0] * 60 + nums[1]
    if len(nums) == 3: return nums[0] * 3600 + nums[1] * 60 + nums[2]
    raise ValueError(f"Cannot parse: {ts!r}")

def _seconds_to_srt(t):
    ms = int(round(t * 1000))
    h,  rem = divmod(ms, 3_600_000)
    m,  rem = divmod(rem,    60_000)
    s,  ms  = divmod(rem,     1_000)
    return f"{h:02d}:{m:02d}:{s:02d},{ms:03d}"

def _txt_to_srt(txt_path: Path) -> Path:
    entries, counter = [], 1
    for line in txt_path.read_text(encoding="utf-8").splitlines():
        mat = _ts_line.match(line.strip())
        if not mat:
            continue
        start_raw, end_raw, text = mat.groups()
        text = text.strip()
        if not text:
            continue
        entries.append(
            f"{counter}\n{_seconds_to_srt(_ts_to_seconds(start_raw))} --> "
            f"{_seconds_to_srt(_ts_to_seconds(end_raw))}\n{text}\n"
        )
        counter += 1
    if not entries:
        raise RuntimeError(f"No [MM:SS-MM:SS] lines found in {txt_path.name}. Check the transcript format.")
    srt_path = txt_path.with_suffix(".srt")
    srt_path.write_text("\n".join(entries), encoding="utf-8")
    print(f"  ✓ {srt_path.name}  ({counter - 1} captions)")
    return srt_path

# ── Shift SRT by the intro duration ───────────────────────────────────────────
_srt_time = re.compile(r"(\d{2,}):(\d{2}):(\d{2}),(\d{3})")

def _shift_srt_text(text: str, offset_sec: float) -> str:
    off_ms = int(round(offset_sec * 1000))
    def _repl(m):
        h, mnt, s, ms = (int(g) for g in m.groups())
        t = h * 3_600_000 + mnt * 60_000 + s * 1_000 + ms + off_ms
        h, rem = divmod(t, 3_600_000)
        mnt, rem = divmod(rem, 60_000)
        s, ms = divmod(rem, 1_000)
        return f"{h:02d}:{mnt:02d}:{s:02d},{ms:03d}"
    return _srt_time.sub(_repl, text)

# ── Render one pair ───────────────────────────────────────────────────────────
def _render_final(audio: Path, srt: Path, out_final: Path) -> Path:
    audio_dur = _duration(audio)

    # Last background loop plays out fully instead of being trimmed to the audio.
    bg_loops  = max(1, ceil(audio_dur / bg_dur))
    total_dur = intro_dur + bg_loops * bg_dur
    fade_dur  = min(FADE_SECONDS, total_dur)

    print(f"  Audio {audio_dur:.2f}s → {bg_loops} bg loops, total {total_dur:.2f}s (fade last {fade_dur:.1f}s)")

    # subtitles filter chokes on Windows absolute paths → work in the audio's
    # folder with a relative .srt name (the output path can be absolute).
    work_dir  = audio.parent
    shifted_srt = work_dir / "_final_subs.srt"
    shifted_srt.write_text(_shift_srt_text(srt.read_text(encoding="utf-8"), intro_dur), encoding="utf-8")

    _scale = (
        f"scale={TARGET_W}:{TARGET_H}:force_original_aspect_ratio=decrease,"
        f"pad={TARGET_W}:{TARGET_H}:-1:-1:color=black,setsar=1,fps={TARGET_FPS}"
    )
    filter_complex = (
        f"[0:v]{_scale}[iv];"
        f"[1:v]{_scale}[bv];"
        f"[iv][bv]concat=n=2:v=1:a=0[v0];"
        f"[v0]subtitles=_final_subs.srt:force_style='{SUBTITLE_STYLE}'[v1];"
        f"[v1]fade=t=out:st={total_dur - fade_dur:.3f}:d={fade_dur:.3f}[vout];"
        f"[0:a]aresample=48000,aformat=channel_layouts=stereo[ia];"
        f"[2:a]aresample=48000,aformat=channel_layouts=stereo[ma];"
        f"[ia][ma]concat=n=2:v=0:a=1[a0];"
        f"[a0]apad=whole_dur={total_dur:.3f}[aout]"
    )

    cmd = [
        str(_ffmpeg), "-y",
        "-i", str(_intro),
        "-stream_loop", str(bg_loops - 1), "-i", str(_bg),
        "-i", str(audio),
        "-filter_complex", filter_complex,
        "-map", "[vout]", "-map", "[aout]",
        *_encode_args,
        "-c:a", "aac", "-b:a", AUDIO_BITRATE,
        "-t", f"{total_dur:.3f}",
        "-movflags", "+faststart",
        str(out_final),
    ]

    print(f"  Encoding with {_encoder} → {out_final.name} …")
    try:
        result = subprocess.run(cmd, cwd=work_dir, capture_output=True, text=True)
        if result.returncode != 0:
            print("  ffmpeg failed:\n", result.stderr[-4000:])
            raise RuntimeError(f"ffmpeg exited with code {result.returncode}")
    finally:
        shifted_srt.unlink(missing_ok=True)

    print(f"  ✓ Wrote {out_final}  ({out_final.stat().st_size / 1e6:.1f} MB)")
    return out_final

# ── Process every pair in succession ──────────────────────────────────────────
done, skipped, failed = [], [], []

for idx, (txt, media) in enumerate(pairs, 1):
    print(f"\n{'─' * 70}\n[{idx}/{len(pairs)}] {media.name}")
    out_final = output_dir / f"{media.stem}_final.mp4"

    if out_final.exists() and not OVERWRITE_FINAL:
        print(f"  ↷ Skipped — already exists: {out_final.name}")
        skipped.append(media.name)
        continue

    try:
        srt = _txt_to_srt(txt)
        _render_final(media, srt, out_final)
        done.append(media.name)
    except Exception as e:
        print(f"  ✗ FAILED: {e}")
        failed.append(media.name)

# ── Summary ───────────────────────────────────────────────────────────────────
print(f"\n{'═' * 70}")
print(f"✓ Rendered: {len(done)}   ↷ Skipped: {len(skipped)}   ✗ Failed: {len(failed)}")
for name in failed:
    print(f"  ✗ {name}")
print(f"Final videos saved in: {output_dir}")

Media folder      : C:\Users\subha\Desktop\transcribe\youtube-translator\input
Transcript folder : C:\Users\subha\Desktop\transcribe\youtube-translator\transcript
Output folder     : C:\Users\subha\Desktop\transcribe\youtube-translator\output
Pairs  : 1
  • truth.txt  ↔  truth.mp3
ffmpeg : C:\ffmpeg\bin\ffmpeg.exe
         ffmpeg version 7.1.1-full_build-www.gyan.dev Copyright (c) 2000-2025 the FFmpeg developers
Encoder : hevc_nvenc
Intro      : Intro.mp4  (10.01s)
Background : Background.mp4  (8.17s per loop)

──────────────────────────────────────────────────────────────────────
[1/1] truth.mp3
  ✓ truth.srt  (21 captions)
  Audio 89.40s → 11 bg loops, total 99.88s (fade last 1.5s)
  Encoding with hevc_nvenc → truth_final.mp4 …
  ✓ Wrote C:\Users\subha\Desktop\transcribe\youtube-translator\output\truth_final.mp4  (64.5 MB)

══════════════════════════════════════════════════════════════════════
✓ Rendered: 1   ↷ Skipped: 0   ✗ Failed: 0
Final videos saved in: C:\Users\subha\Desktop\tr